**Notebook Cell 1: Introduction**


# Time Series Analysis of AI Adoption in Australian Construction Industry

## Research Overview
This notebook analyzes AI-related job postings in the Australian construction industry from [redacted_phone], examining:
- Temporal evolution and growth patterns
- Seasonal variations
- Cross-industry comparisons
- Lead-lag relationships with other sectors

**Dataset:** Australian job postings across all industries ([redacted_phone])  
**Focus:** Construction industry (ANZSIC codes [redacted_phone])  
**Author:** Dr. M. Reza Hosseini  
**Date:** January 2025


**Notebook Cell 2: Environment Setup**


In [ ]:
# ============================================
# SECTION 1: ENVIRONMENT SETUP
# ============================================
"""
This section sets up the working environment, imports necessary libraries,
and configures display settings for optimal data exploration.
"""

# Core library imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import os

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Configure pandas display options for better data viewing
pd.set_option('display.max_columns', 20)  # Show up to 20 columns
pd.set_option('display.max_rows', 100)     # Show up to 100 rows
pd.set_option('display.width', 1000)       # Wider display
pd.set_option('display.precision', 2)      # 2 decimal places for floats

# Set matplotlib style for better visualizations
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("=" * 60)
print("ENVIRONMENT SETUP COMPLETE")
print("=" * 60)
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

# ============================================
# SECTION 2: GOOGLE DRIVE MOUNT & PATH SETUP
# ============================================
"""
Mount Google Drive and establish the data path.
This ensures access to the dataset stored in Drive.
"""

from google.colab import drive
drive.mount('YOUR_PATH')

# Define the data path
DATA_PATH = 'YOUR_PATH'
FILE_NAME = 'all_industries_merged.xlsx'
FULL_PATH = DATA_PATH + FILE_NAME

# Verify file exists
if os.path.exists(FULL_PATH):
    file_size = os.path.getsize(FULL_PATH) / (1024 * 1024)  # Convert to MB
    print(f"\n✅ File found: {FILE_NAME}")
    print(f"📁 Location: {DATA_PATH}")
    print(f"📊 File size: {file_size:.2f} MB")
else:
    print(f"\n❌ File not found at: {FULL_PATH}")
    print("Please check the path and file name.")

print("\n" + "=" * 60)

# ============================================
# CHECKPOINT 1: INITIAL DATA LOADING
# ============================================
"""
Load the Excel file and perform initial inspection.
This checkpoint verifies successful data loading and provides
basic information about the dataset structure.
"""

print("\n📍 CHECKPOINT 1: LOADING DATA")
print("=" * 60)

try:
    # Load the Excel file
    print("Loading Excel file... (this may take a moment for large files)")
    df = pd.read_excel(FULL_PATH)

    # Basic dataset information
    print(f"\n✅ Data loaded successfully!")
    print(f"\n📊 DATASET DIMENSIONS:")
    print(f"   - Total records: {len(df):,}")
    print(f"   - Total columns: {len(df.columns)}")
    print(f"   - Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

    # Column information
    print(f"\n📋 COLUMN NAMES AND TYPES:")
    print("-" * 40)
    for col in df.columns:
        print(f"   {col}: {df[col].dtype}")

    # First few rows
    print(f"\n👀 FIRST 3 ROWS OF DATA:")
    print("-" * 40)
    print(df.head(3))

except Exception as e:
    print(f"\n❌ Error loading file: {str(e)}")
    print("Please check if the file exists and is a valid Excel file.")

print("\n" + "=" * 60)

# ============================================
# CHECKPOINT 2: DATA QUALITY ASSESSMENT
# ============================================
"""
Assess data quality by checking for missing values,
data types, and basic statistics for key columns.
"""

print("\n📍 CHECKPOINT 2: DATA QUALITY ASSESSMENT")
print("=" * 60)

if 'df' in locals():
    # Missing values analysis
    print("\n🔍 MISSING VALUES ANALYSIS:")
    print("-" * 40)
    missing_df = pd.DataFrame({
        'Column': df.columns,
        'Missing_Count': df.isnull().sum(),
        'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
    })
    missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Percentage', ascending=False)

    if len(missing_df) > 0:
        print(missing_df.to_string(index=False))
    else:
        print("✅ No missing values found in any column!")

    # Check for date columns
    print("\n📅 DATE COLUMN ANALYSIS:")
    print("-" * 40)
    date_columns = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]

    if date_columns:
        for col in date_columns:
            print(f"\nColumn: {col}")
            print(f"   Type: {df[col].dtype}")
            if df[col].dtype == 'object':
                print(f"   Sample values: {df[col].dropna().head(3).tolist()}")
            else:
                try:
                    print(f"   Min value: {df[col].min()}")
                    print(f"   Max value: {df[col].max()}")
                except:
                    print(f"   Could not analyze - check data type")
    else:
        print("⚠️ No obvious date columns found (looking for 'date' or 'time' in column names)")

    # Industry code analysis (ANZSIC)
    print("\n🏭 INDUSTRY CODE ANALYSIS:")
    print("-" * 40)
    anzsic_columns = [col for col in df.columns if 'anzsic' in col.lower()]

    if anzsic_columns:
        for col in anzsic_columns:
            print(f"\nColumn: {col}")
            print(f"   Unique values: {df[col].nunique()}")
            print(f"   Data type: {df[col].dtype}")

            # Check for construction codes ([redacted_phone])
            if pd.api.types.is_numeric_dtype(df[col]):
                construction_mask = df[col].between(3000, 3999)
                construction_count = construction_mask.sum()
                print(f"   Construction industry records ([redacted_phone]): {construction_count:,}")
                print(f"   Construction percentage: {(construction_count/len(df)*100):.2%}")
    else:
        print("⚠️ No ANZSIC code column found")

    # Basic statistics for numeric columns
    print("\n📊 NUMERIC COLUMNS SUMMARY:")
    print("-" * 40)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        print(f"Found {len(numeric_cols)} numeric columns")
        print(df[numeric_cols].describe().round(2))
    else:
        print("No numeric columns found")

print("\n" + "=" * 60)

# ============================================
# CHECKPOINT 3: TEMPORAL COVERAGE ANALYSIS
# ============================================
"""
Analyze the temporal coverage of the dataset to verify
it spans [redacted_phone] and understand the distribution of records over time.
"""

print("\n📍 CHECKPOINT 3: TEMPORAL COVERAGE ANALYSIS")
print("=" * 60)

if 'df' in locals() and date_columns:
    print("\n⏰ CONVERTING DATE COLUMNS...")

    # Try to convert the first date column found
    date_col = date_columns[0]
    print(f"Using column: {date_col}")

    try:
        # Check if dates are in Excel serial number format
        if pd.api.types.is_numeric_dtype(df[date_col]):
            # Excel serial dates (days since [redacted_phone], with leap year bug)
            df['parsed_date'] = pd.to_datetime('[redacted_phone]') + pd.to_timedelta(df[date_col], 'D')
            print("✅ Converted from Excel serial date format")
        else:
            # Try standard date parsing
            df['parsed_date'] = pd.to_datetime(df[date_col])
            print("✅ Converted from standard date format")

        # Extract temporal components
        df['year'] = df['parsed_date'].dt.year
        df['month'] = df['parsed_date'].dt.month
        df['quarter'] = df['parsed_date'].dt.quarter
        df['year_month'] = df['parsed_date'].dt.to_period('M')
        df['year_quarter'] = df['parsed_date'].dt.to_period('Q')

        # Temporal coverage summary
        print(f"\n📆 TEMPORAL COVERAGE:")
        print("-" * 40)
        print(f"   Date range: {df['parsed_date'].min()} to {df['parsed_date'].max()}")
        print(f"   Total days covered: {(df['parsed_date'].max() - df['parsed_date'].min()).days}")
        print(f"   Unique years: {sorted(df['year'].unique())}")

        # Records per year
        print(f"\n📊 RECORDS PER YEAR:")
        print("-" * 40)
        year_counts = df['year'].value_counts().sort_index()
        for year, count in year_counts.items():
            print(f"   {year}: {count:,} records ({count/len(df)*100:.1f}%)")

        # Monthly distribution
        print(f"\n📊 AVERAGE RECORDS PER MONTH:")
        print("-" * 40)
        monthly_avg = df.groupby('year_month').size().mean()
        print(f"   Average: {monthly_avg:.0f} records/month")
        print(f"   Total months with data: {df['year_month'].nunique()}")

    except Exception as e:
        print(f"\n⚠️ Could not parse dates: {str(e)}")
        print("Please check the date format in your data")

print("\n" + "=" * 60)
print("\n✅ INITIAL EXPLORATION COMPLETE")
print("=" * 60)
print("\nPlease share the output above, especially:")
print("1. Whether the data loaded successfully")
print("2. The actual column names in your dataset")
print("3. The date range and temporal coverage")
print("4. The number of construction industry records")
print("\nBased on this information, we'll proceed with the next steps.")


# Step 2: Temporal Distribution Analysis & Industry Classification


In [ ]:
# ============================================
# STEP 2: TEMPORAL DISTRIBUTION & INDUSTRY CLASSIFICATION
# ============================================
"""
This section examines temporal distribution (especially 2025)
and classifies industries into Digital vs Traditional categories
"""

print("=" * 60)
print("STEP 2: TEMPORAL DISTRIBUTION & INDUSTRY CLASSIFICATION")
print("=" * 60)

# ============================================
# CHECKPOINT 4: 2025 DATA EXAMINATION
# ============================================
"""
Examine the distribution of 2025 data to make informed decision
about inclusion/exclusion in time series analysis
"""

print("\n📍 CHECKPOINT 4: EXAMINING 2025 DATA DISTRIBUTION")
print("=" * 60)

# Monthly distribution for all years with focus on 2025
monthly_dist = df.groupby(['year', 'month']).size().reset_index(name='count')

# Pivot for better visualization
monthly_pivot = monthly_dist.pivot(index='month', columns='year', values='count').fillna(0)

print("\n📊 MONTHLY DISTRIBUTION BY YEAR:")
print("-" * 40)
print(monthly_pivot.astype(int))

# Specific 2025 analysis
df_2025 = df[df['year'] == 2025].copy()
print(f"\n🔍 2025 DATA ANALYSIS:")
print("-" * 40)
print(f"Total 2025 records: {len(df_2025):,}")

# Monthly breakdown for 2025
monthly_2025 = df_2025.groupby('month').size().sort_index()
print(f"\n2025 Monthly breakdown:")
for month, count in monthly_2025.items():
    month_name = pd.Timestamp(2025, month, 1).strftime('%B')
    print(f"   {month_name}: {count:,} records")

# Check last date in each month of 2025
print(f"\n📅 Latest date in each 2025 month:")
for month in monthly_2025.index:
    month_data = df_2025[df_2025['month'] == month]
    latest = month_data['parsed_date'].max()
    print(f"   Month {month}: Last record on {latest.strftime('%Y-%m-%d')}")

# Statistical comparison
print(f"\n📊 STATISTICAL COMPARISON:")
print("-" * 40)
avg_2024_monthly = df[df['year'] == 2024].groupby('month').size().mean()
avg_2025_monthly = monthly_2025.mean()
print(f"Average monthly records 2024: {avg_2024_monthly:.0f}")
print(f"Average monthly records 2025: {avg_2025_monthly:.0f}")
print(f"2025 vs 2024 ratio: {avg_2025_monthly/avg_2024_monthly:.2%}")

print("\n" + "=" * 60)

# ============================================
# CHECKPOINT 5: INDUSTRY CLASSIFICATION
# ============================================
"""
Classify industries into Digital vs Traditional based on ANZSIC codes
ANZSIC Industry Divisions:
- [redacted_phone]: Construction (Traditional)
- [redacted_phone]: Retail Trade (Traditional)
- [redacted_phone]: Accommodation, Transport (Traditional)
- [redacted_phone]: IT, Media, Finance (Digital)
- [redacted_phone]: Professional Services (Mixed - will classify as Digital)
- [redacted_phone]: Healthcare, Education (Traditional)
- [redacted_phone]: Other Services (Traditional)
"""

print("\n📍 CHECKPOINT 5: INDUSTRY CLASSIFICATION (DIGITAL vs TRADITIONAL)")
print("=" * 60)

def classify_industry(anzsic_code):
    """
    Classify industry as Digital or Traditional based on ANZSIC code
    Digital: IT, Media, Finance, Professional Services
    Traditional: Construction, Manufacturing, Retail, Healthcare, Education
    """
    if pd.isna(anzsic_code):
        return 'Unknown'

    code = int(anzsic_code)

    # Digital industries (knowledge/tech intensive)
    if 5800 <= code <= 6399:  # Information Media, Telecommunications, Finance
        return 'Digital'
    elif 6900 <= code <= 7299:  # Professional, Scientific, Technical Services
        return 'Digital'

    # Traditional industries
    elif 3000 <= code <= 3999:  # Construction
        return 'Traditional-Construction'
    elif 1000 <= code <= 2999:  # Agriculture, Mining, Manufacturing
        return 'Traditional-Other'
    elif 4000 <= code <= 4999:  # Retail Trade
        return 'Traditional-Other'
    elif 5000 <= code <= 5799:  # Accommodation, Transport, Warehousing
        return 'Traditional-Other'
    elif 6400 <= code <= 6899:  # Rental, Real Estate
        return 'Traditional-Other'
    elif 7300 <= code <= 7999:  # Public Administration, Safety
        return 'Traditional-Other'
    elif 8000 <= code <= 8999:  # Healthcare, Education
        return 'Traditional-Other'
    elif 9000 <= code <= 9999:  # Other Services
        return 'Traditional-Other'
    else:
        return 'Other'

# Apply classification
df['industry_type'] = df['ANZSIC CODE'].apply(classify_industry)
df['is_construction'] = df['ANZSIC CODE'].between(3000, 3999)

# Summary statistics
print("\n📊 INDUSTRY CLASSIFICATION RESULTS:")
print("-" * 40)
industry_counts = df['industry_type'].value_counts()
for industry, count in industry_counts.items():
    percentage = (count / len(df)) * 100
    print(f"{industry:25s}: {count:6,} records ({percentage:5.1f}%)")

# Separate Digital vs Traditional (combining construction with other traditional)
df['sector_type'] = df['industry_type'].apply(
    lambda x: 'Digital' if x == 'Digital' else
              ('Traditional' if 'Traditional' in x else 'Other')
)

print("\n📊 SECTOR SUMMARY (DIGITAL vs TRADITIONAL):")
print("-" * 40)
sector_counts = df['sector_type'].value_counts()
for sector, count in sector_counts.items():
    percentage = (count / len(df)) * 100
    print(f"{sector:15s}: {count:6,} records ({percentage:5.1f}%)")

# Construction within Traditional
traditional_df = df[df['sector_type'] == 'Traditional']
construction_in_traditional = df['is_construction'].sum()
print(f"\n🏗️ CONSTRUCTION WITHIN TRADITIONAL SECTOR:")
print("-" * 40)
print(f"Construction records: {construction_in_traditional:,}")
print(f"Total traditional records: {len(traditional_df):,}")
print(f"Construction as % of traditional: {(construction_in_traditional/len(traditional_df)*100):.1f}%")

print("\n" + "=" * 60)

# ============================================
# CHECKPOINT 6: TEMPORAL AGGREGATION
# ============================================
"""
Create time series for different industry groups
"""

print("\n📍 CHECKPOINT 6: TEMPORAL AGGREGATION BY SECTOR")
print("=" * 60)

# Create quarterly aggregations for more robust analysis
df['year_quarter'] = df['parsed_date'].dt.to_period('Q')

# Quarterly counts by sector
quarterly_sectors = df.groupby(['year_quarter', 'sector_type']).size().unstack(fill_value=0)
quarterly_construction = df[df['is_construction']].groupby('year_quarter').size()

print("\n📊 QUARTERLY AI JOB COUNTS BY SECTOR:")
print("-" * 40)
print(quarterly_sectors.head(8))  # Show first 2 years

# Calculate quarter-on-quarter growth rates
print("\n📈 QUARTER-ON-QUARTER GROWTH RATES (%):")
print("-" * 40)
growth_rates = quarterly_sectors.pct_change() * 100
print(growth_rates.iloc[1:8].round(1))  # Skip first row (NaN) and show 2 years

# Construction specific quarterly data
print("\n🏗️ CONSTRUCTION QUARTERLY TRENDS:")
print("-" * 40)
construction_quarterly = df[df['is_construction']].groupby('year_quarter').size()
for quarter in construction_quarterly.index[:8]:  # First 2 years
    count = construction_quarterly[quarter]
    print(f"{quarter}: {count} AI jobs")

# Average AI jobs per quarter by sector
print("\n📊 AVERAGE QUARTERLY AI JOBS ([redacted_phone]):")
print("-" * 40)
# Exclude 2025 for fair comparison
df_2021_2024 = df[df['year'] < 2025]
avg_quarterly = df_2021_2024.groupby('sector_type').size() / 16  # 16 quarters in 4 years
for sector, avg in avg_quarterly.items():
    print(f"{sector:15s}: {avg:.0f} jobs/quarter")

# Construction average
construction_avg = df_2021_2024[df_2021_2024['is_construction']].shape[0] / 16
print(f"{'Construction':15s}: {construction_avg:.0f} jobs/quarter")

print("\n" + "=" * 60)

# ============================================
# RECOMMENDATION FOR 2025 DATA
# ============================================

print("\n💡 RECOMMENDATION FOR 2025 DATA HANDLING:")
print("=" * 60)

# Calculate completeness of Q1 2025
q1_2025_days = (df_2025['parsed_date'].max() - pd.Timestamp('[redacted_phone]')).days
q1_total_days = 90  # Jan-Mar
completeness = (q1_2025_days / q1_total_days) * 100

print(f"\n2025 Q1 Completeness: {completeness:.1f}%")
print(f"Days covered: {q1_2025_days} out of {q1_total_days}")

if completeness >= 85:
    print("\n✅ RECOMMENDATION: Include Q1 2025")
    print("   - Q1 is nearly complete (>85%)")
    print("   - Can be used for trend analysis with notation")
elif completeness >= 50:
    print("\n⚠️ RECOMMENDATION: Include with caution")
    print("   - Q1 is partially complete (50-85%)")
    print("   - Adjust for incompleteness or exclude from forecasting")
else:
    print("\n❌ RECOMMENDATION: Exclude 2025")
    print("   - Q1 is incomplete (<50%)")
    print("   - Use [redacted_phone] for main analysis")

print("\n" + "=" * 60)
print("✅ INDUSTRY CLASSIFICATION AND TEMPORAL ANALYSIS COMPLETE")
print("=" * 60)


**Step 3.1: Trend Analysis and Long-term Patterns**


In [ ]:
# ============================================
# COMPLETE STEP 3.1: TREND ANALYSIS (SELF-CONTAINED)
# ============================================
"""
Complete trend analysis code with corrected x-axis labels
This is self-contained and includes all necessary data preparation
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.filters.hp_filter import hpfilter
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("COMPLETE TREND ANALYSIS WITH CORRECTED FIGURES")
print("=" * 60)

# ============================================
# DATA PREPARATION
# ============================================

print("\n📊 PREPARING TIME SERIES DATA")
print("-" * 40)

# Filter for complete quarters (2021 Q1 - 2024 Q4)
df_analysis = df[df['year'] <= 2024].copy()

# Create quarterly aggregations
quarterly_digital = df_analysis[df_analysis['sector_type'] == 'Digital'].groupby('year_quarter').size()
quarterly_traditional = df_analysis[df_analysis['sector_type'] == 'Traditional'].groupby('year_quarter').size()
quarterly_construction = df_analysis[df_analysis['is_construction']].groupby('year_quarter').size()

# Ensure all series have same index
quarters = pd.period_range('2021Q1', '2024Q4', freq='Q')
quarterly_digital = quarterly_digital.reindex(quarters, fill_value=0)
quarterly_traditional = quarterly_traditional.reindex(quarters, fill_value=0)
quarterly_construction = quarterly_construction.reindex(quarters, fill_value=0)

print(f"Time series prepared for {len(quarters)} quarters")

# ============================================
# HODRICK-PRESCOTT FILTER
# ============================================

print("\n📈 APPLYING HODRICK-PRESCOTT FILTER")
print("-" * 40)

# Apply HP filter
digital_cycle, digital_trend = hpfilter(quarterly_digital, lamb=1600)
traditional_cycle, traditional_trend = hpfilter(quarterly_traditional, lamb=1600)
construction_cycle, construction_trend = hpfilter(quarterly_construction, lamb=1600)

print("HP filter applied successfully")

# ============================================
# GROWTH CALCULATIONS
# ============================================

print("\n📊 CALCULATING GROWTH METRICS")
print("-" * 40)

# Growth indices
base_digital = quarterly_digital.iloc[0]
base_traditional = quarterly_traditional.iloc[0]
base_construction = quarterly_construction.iloc[0]

digital_index = (quarterly_digital / base_digital * 100).round(1)
traditional_index = (quarterly_traditional / base_traditional * 100).round(1)
construction_index = (quarterly_construction / base_construction * 100).round(1)

# CAGR
years = 4
digital_cagr = ((quarterly_digital.iloc[-1] / quarterly_digital.iloc[0]) ** (1/years) - 1) * 100
traditional_cagr = ((quarterly_traditional.iloc[-1] / quarterly_traditional.iloc[0]) ** (1/years) - 1) * 100
construction_cagr = ((quarterly_construction.iloc[-1] / quarterly_construction.iloc[0]) ** (1/years) - 1) * 100

print(f"Digital CAGR: {digital_cagr:+.1f}%")
print(f"Traditional CAGR: {traditional_cagr:+.1f}%")
print(f"Construction CAGR: {construction_cagr:+.1f}%")

# ============================================
# FIGURE 1: CORRECTED VISUALIZATION
# ============================================

print("\n📊 CREATING FIGURE 1: TREND ANALYSIS (CORRECTED)")
print("-" * 40)

# Set style for publication
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Convert quarters to datetime for plotting
x_dates = pd.to_datetime(quarters.to_timestamp())

# Panel A: Digital Sector
ax1 = axes[0, 0]
ax1.plot(x_dates, quarterly_digital.values, 'o-', color='#2E86AB', alpha=0.6, label='Actual', markersize=6)
ax1.plot(x_dates, digital_trend, '-', color='#A23B72', linewidth=2.5, label='HP Trend')
ax1.set_title('(a) Digital Sector', fontsize=12, fontweight='bold')
ax1.set_xlabel('Quarter', fontsize=11)
ax1.set_ylabel('Number of AI Job Postings', fontsize=11)
ax1.legend(loc='upper left', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.annotate(f'CAGR: {digital_cagr:+.1f}%',
             xy=(0.95, 0.05), xycoords='axes fraction',
             ha='right', fontsize=10,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# Panel B: Traditional Sector
ax2 = axes[0, 1]
ax2.plot(x_dates, quarterly_traditional.values, 'o-', color='#F18F01', alpha=0.6, label='Actual', markersize=6)
ax2.plot(x_dates, traditional_trend, '-', color='#C73E1D', linewidth=2.5, label='HP Trend')
ax2.set_title('(b) Traditional Sector', fontsize=12, fontweight='bold')
ax2.set_xlabel('Quarter', fontsize=11)
ax2.set_ylabel('Number of AI Job Postings', fontsize=11)
ax2.legend(loc='upper left', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.annotate(f'CAGR: {traditional_cagr:+.1f}%',
             xy=(0.95, 0.05), xycoords='axes fraction',
             ha='right', fontsize=10,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# Panel C: Construction Industry
ax3 = axes[1, 0]
ax3.plot(x_dates, quarterly_construction.values, 'o-', color='#6B8E23', alpha=0.6, label='Actual', markersize=6)
ax3.plot(x_dates, construction_trend, '-', color='#8B4513', linewidth=2.5, label='HP Trend')
ax3.set_title('(c) Construction Industry', fontsize=12, fontweight='bold')
ax3.set_xlabel('Quarter', fontsize=11)
ax3.set_ylabel('Number of AI Job Postings', fontsize=11)
ax3.legend(loc='upper left', fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.annotate(f'CAGR: {construction_cagr:+.1f}%',
             xy=(0.95, 0.05), xycoords='axes fraction',
             ha='right', fontsize=10,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# Panel D: Growth Indices Comparison
ax4 = axes[1, 1]
ax4.plot(x_dates, digital_index.values, '-', color='#2E86AB', linewidth=2.5, label='Digital', marker='o', markersize=5)
ax4.plot(x_dates, traditional_index.values, '-', color='#F18F01', linewidth=2.5, label='Traditional', marker='s', markersize=5)
ax4.plot(x_dates, construction_index.values, '-', color='#6B8E23', linewidth=2.5, label='Construction', marker='^', markersize=5)
ax4.set_title('(d) Growth Indices Comparison (Q1 2021 = 100)', fontsize=12, fontweight='bold')
ax4.set_xlabel('Quarter', fontsize=11)
ax4.set_ylabel('Growth Index', fontsize=11)
ax4.legend(loc='upper left', fontsize=10)
ax4.grid(True, alpha=0.3)
ax4.axhline(y=100, color='gray', linestyle='--', alpha=0.5)

# Fix x-axis for all subplots
for ax in axes.flat:
    ax.tick_params(axis='both', labelsize=10)

    # Set x-ticks at quarterly intervals (show every 2nd quarter)
    ax.set_xticks(x_dates[::2])

    # Create custom labels showing Year Q#
    tick_labels = []
    for i in range(0, len(quarters), 2):
        year = quarters[i].year
        quarter = quarters[i].quarter
        tick_labels.append(f'{year}Q{quarter}')

    ax.set_xticklabels(tick_labels, rotation=45, ha='right')

plt.suptitle('AI Job Posting Trends by Sector ([redacted_phone])', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figure1_trend_analysis_corrected.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Figure 1 saved as 'figure1_trend_analysis_corrected.png'")

# ============================================
# TABLE 1: TREND ANALYSIS SUMMARY
# ============================================

print("\n" + "=" * 60)
print("TABLE 1: TREND ANALYSIS SUMMARY")
print("=" * 60)

# Create summary statistics table
summary_data = {
    'Sector': ['Digital', 'Traditional', 'Construction'],
    'Total Jobs ([redacted_phone])': [
        quarterly_digital.sum(),
        quarterly_traditional.sum(),
        quarterly_construction.sum()
    ],
    'Q1 2021': [
        quarterly_digital.iloc[0],
        quarterly_traditional.iloc[0],
        quarterly_construction.iloc[0]
    ],
    'Q4 2024': [
        quarterly_digital.iloc[-1],
        quarterly_traditional.iloc[-1],
        quarterly_construction.iloc[-1]
    ],
    'CAGR (%)': [
        digital_cagr,
        traditional_cagr,
        construction_cagr
    ],
    'Growth Index Q4 2024': [
        digital_index.iloc[-1],
        traditional_index.iloc[-1],
        construction_index.iloc[-1]
    ],
    'Avg Quarterly Growth (%)': [
        quarterly_digital.pct_change().mean()*100,
        quarterly_traditional.pct_change().mean()*100,
        quarterly_construction.pct_change().mean()*100
    ]
}

summary_table = pd.DataFrame(summary_data)

# Format the table for display
display_table = summary_table.copy()
display_table['Total Jobs ([redacted_phone])'] = display_table['Total Jobs ([redacted_phone])'].apply(lambda x: f'{x:,}')
display_table['CAGR (%)'] = display_table['CAGR (%)'].apply(lambda x: f'{x:+.1f}')
display_table['Growth Index Q4 2024'] = display_table['Growth Index Q4 2024'].apply(lambda x: f'{x:.1f}')
display_table['Avg Quarterly Growth (%)'] = display_table['Avg Quarterly Growth (%)'].apply(lambda x: f'{x:+.1f}')

print("\n", display_table.to_string(index=False))

# Save table to CSV
summary_table.to_csv('table1_trend_summary.csv', index=False)
print("\n✅ Table saved as 'table1_trend_summary.csv'")

# ============================================
# MANN-KENDALL TREND TEST
# ============================================

print("\n" + "=" * 60)
print("MANN-KENDALL TREND TEST RESULTS")
print("=" * 60)

from scipy.stats import kendalltau

def mann_kendall_test(data):
    """Simplified Mann-Kendall trend test"""
    n = len(data)
    x = np.arange(1, n+1)
    tau, p_value = kendalltau(x, data)

    # Determine trend
    if p_value < 0.05:
        if tau > 0:
            trend = "Increasing (significant)"
        else:
            trend = "Decreasing (significant)"
    else:
        trend = "No significant trend"

    return tau, p_value, trend

# Test each sector
digital_tau, digital_p, digital_trend = mann_kendall_test(quarterly_digital.values)
traditional_tau, traditional_p, traditional_trend = mann_kendall_test(quarterly_traditional.values)
construction_tau, construction_p, construction_trend = mann_kendall_test(quarterly_construction.values)

print(f"\nDigital Sector:")
print(f"  Kendall's tau: {digital_tau:.3f}, p-value: {digital_p:.4f}")
print(f"  Trend: {digital_trend}")

print(f"\nTraditional Sector:")
print(f"  Kendall's tau: {traditional_tau:.3f}, p-value: {traditional_p:.4f}")
print(f"  Trend: {traditional_trend}")

print(f"\nConstruction Industry:")
print(f"  Kendall's tau: {construction_tau:.3f}, p-value: {construction_p:.4f}")
print(f"  Trend: {construction_trend}")

print("\n" + "=" * 60)
print("✅ TREND ANALYSIS COMPLETE")
print("=" * 60)
print("\nPlease share these results before proceeding to seasonality analysis.")


**Seasonality Analysis**


In [ ]:
# ============================================
# STEP 3.2: SEASONALITY ANALYSIS
# ============================================
"""
Comprehensive seasonality analysis using STL decomposition,
seasonal strength metrics, and statistical tests
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import STL
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("STEP 3.2: SEASONALITY ANALYSIS")
print("=" * 60)

# ============================================
# DATA PREPARATION FOR SEASONALITY
# ============================================

print("\n📊 PREPARING DATA FOR SEASONAL ANALYSIS")
print("-" * 40)

# Recreate the quarterly time series ([redacted_phone])
df_analysis = df[df['year'] <= 2024].copy()

# Create quarterly aggregations
quarterly_digital = df_analysis[df_analysis['sector_type'] == 'Digital'].groupby('year_quarter').size()
quarterly_traditional = df_analysis[df_analysis['sector_type'] == 'Traditional'].groupby('year_quarter').size()
quarterly_construction = df_analysis[df_analysis['is_construction']].groupby('year_quarter').size()

# Ensure proper indexing
quarters = pd.period_range('2021Q1', '2024Q4', freq='Q')
quarterly_digital = quarterly_digital.reindex(quarters, fill_value=0)
quarterly_traditional = quarterly_traditional.reindex(quarters, fill_value=0)
quarterly_construction = quarterly_construction.reindex(quarters, fill_value=0)

# Convert to proper time series with datetime index for STL
dates = pd.date_range(start='[redacted_phone]', periods=16, freq='Q')
ts_digital = pd.Series(quarterly_digital.values, index=dates)
ts_traditional = pd.Series(quarterly_traditional.values, index=dates)
ts_construction = pd.Series(quarterly_construction.values, index=dates)

print(f"Prepared {len(dates)} quarters of data for analysis")

# ============================================
# STL DECOMPOSITION
# ============================================

print("\n📈 PERFORMING STL DECOMPOSITION")
print("-" * 40)

# Perform STL decomposition for each sector
# Using seasonal=5 for quarterly data (odd number close to 4)
stl_digital = STL(ts_digital, seasonal=5, trend=7).fit()
stl_traditional = STL(ts_traditional, seasonal=5, trend=7).fit()
stl_construction = STL(ts_construction, seasonal=5, trend=7).fit()

print("STL decomposition completed for all sectors")

# ============================================
# SEASONAL STRENGTH CALCULATION
# ============================================

print("\n📊 CALCULATING SEASONAL STRENGTH")
print("-" * 40)

def calculate_seasonal_strength(stl_result):
    """
    Calculate seasonal strength metric
    Strength = 1 - Var(Remainder) / Var(Seasonal + Remainder)
    Values > 0.64 indicate substantial seasonality
    """
    seasonal = stl_result.seasonal
    remainder = stl_result.resid

    var_remainder = np.var(remainder)
    var_seasonal_remainder = np.var(seasonal + remainder)

    if var_seasonal_remainder == 0:
        return 0

    strength = max(0, 1 - var_remainder / var_seasonal_remainder)
    return strength

digital_strength = calculate_seasonal_strength(stl_digital)
traditional_strength = calculate_seasonal_strength(stl_traditional)
construction_strength = calculate_seasonal_strength(stl_construction)

print(f"Seasonal Strength (0-1 scale, >0.64 = substantial):")
print(f"  Digital: {digital_strength:.3f}")
print(f"  Traditional: {traditional_strength:.3f}")
print(f"  Construction: {construction_strength:.3f}")

# Interpretation
for sector, strength in [("Digital", digital_strength),
                         ("Traditional", traditional_strength),
                         ("Construction", construction_strength)]:
    if strength > 0.64:
        interpretation = "Strong seasonality"
    elif strength > 0.4:
        interpretation = "Moderate seasonality"
    else:
        interpretation = "Weak/No seasonality"
    print(f"  {sector}: {interpretation}")

# ============================================
# SEASONAL FACTORS BY QUARTER
# ============================================

print("\n📊 SEASONAL FACTORS BY QUARTER")
print("-" * 40)

def calculate_seasonal_factors(data):
    """Calculate average seasonal factor for each quarter"""
    quarterly_means = []
    for q in range(1, 5):  # Q1 to Q4
        quarter_data = [data[i] for i in range(len(data)) if (i % 4) == (q-1)]
        quarterly_means.append(np.mean(quarter_data))

    # Calculate seasonal factors (multiplicative)
    overall_mean = np.mean(data)
    seasonal_factors = [qm / overall_mean if overall_mean > 0 else 1 for qm in quarterly_means]

    return seasonal_factors

# Calculate seasonal factors
digital_factors = calculate_seasonal_factors(quarterly_digital.values)
traditional_factors = calculate_seasonal_factors(quarterly_traditional.values)
construction_factors = calculate_seasonal_factors(quarterly_construction.values)

# Create seasonal factors table
seasonal_table = pd.DataFrame({
    'Quarter': ['Q1', 'Q2', 'Q3', 'Q4'],
    'Digital': digital_factors,
    'Traditional': traditional_factors,
    'Construction': construction_factors
})

print("\nSeasonal Factors (1.0 = average, >1.0 = above average):")
print(seasonal_table.round(3).to_string(index=False))

# ============================================
# FIGURE 2: SEASONAL DECOMPOSITION
# ============================================

print("\n📊 CREATING FIGURE 2: SEASONAL DECOMPOSITION")
print("-" * 40)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

# Define sectors and their decompositions
sectors = [
    ('Digital', ts_digital, stl_digital, '#2E86AB'),
    ('Traditional', ts_traditional, stl_traditional, '#F18F01'),
    ('Construction', ts_construction, stl_construction, '#6B8E23')
]

for idx, (name, ts_data, stl_result, color) in enumerate(sectors):
    # Original series
    axes[idx, 0].plot(ts_data.index, ts_data.values, color=color, linewidth=2)
    axes[idx, 0].set_title(f'{name}: Original', fontweight='bold')
    axes[idx, 0].set_ylabel('Job Postings')
    axes[idx, 0].grid(True, alpha=0.3)

    # Trend component
    axes[idx, 1].plot(ts_data.index, stl_result.trend, color=color, linewidth=2)
    axes[idx, 1].set_title(f'{name}: Trend', fontweight='bold')
    axes[idx, 1].set_ylabel('Trend')
    axes[idx, 1].grid(True, alpha=0.3)

    # Seasonal component
    axes[idx, 2].plot(ts_data.index, stl_result.seasonal, color=color, linewidth=2)
    axes[idx, 2].set_title(f'{name}: Seasonal', fontweight='bold')
    axes[idx, 2].set_ylabel('Seasonal')
    axes[idx, 2].grid(True, alpha=0.3)
    axes[idx, 2].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

    # Residual component
    axes[idx, 3].plot(ts_data.index, stl_result.resid, color=color, alpha=0.7, linewidth=1)
    axes[idx, 3].scatter(ts_data.index, stl_result.resid, color=color, alpha=0.5, s=20)
    axes[idx, 3].set_title(f'{name}: Residual', fontweight='bold')
    axes[idx, 3].set_ylabel('Residual')
    axes[idx, 3].grid(True, alpha=0.3)
    axes[idx, 3].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Format x-axis
for ax in axes.flat:
    ax.tick_params(axis='both', labelsize=9)
    # Set quarterly x-ticks
    ax.set_xticks(dates[::2])  # Every 2 quarters
    labels = [f'{d.year}Q{((d.month-1)//3)+1}' for d in dates[::2]]
    ax.set_xticklabels(labels, rotation=45, ha='right')

plt.suptitle('STL Decomposition of AI Job Postings by Sector', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figure2_seasonal_decomposition.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Figure 2 saved as 'figure2_seasonal_decomposition.png'")

# ============================================
# FIGURE 3: SEASONAL PATTERNS COMPARISON
# ============================================

print("\n📊 CREATING FIGURE 3: SEASONAL PATTERNS COMPARISON")
print("-" * 40)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel A: Seasonal subseries plot
ax1 = axes[0, 0]
quarters_list = ['Q1', 'Q2', 'Q3', 'Q4']
x_pos = np.arange(len(quarters_list))
width = 0.25

digital_means = [np.mean([quarterly_digital.values[i] for i in range(16) if i % 4 == q])
                 for q in range(4)]
traditional_means = [np.mean([quarterly_traditional.values[i] for i in range(16) if i % 4 == q])
                    for q in range(4)]
construction_means = [np.mean([quarterly_construction.values[i] for i in range(16) if i % 4 == q])
                     for q in range(4)]

ax1.bar(x_pos - width, digital_means, width, label='Digital', color='#2E86AB', alpha=0.8)
ax1.bar(x_pos, traditional_means, width, label='Traditional', color='#F18F01', alpha=0.8)
ax1.bar(x_pos + width, construction_means, width, label='Construction', color='#6B8E23', alpha=0.8)

ax1.set_title('(a) Average Job Postings by Quarter', fontsize=12, fontweight='bold')
ax1.set_xlabel('Quarter', fontsize=11)
ax1.set_ylabel('Average Number of Jobs', fontsize=11)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(quarters_list)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# Panel B: Seasonal factors heatmap
ax2 = axes[0, 1]
seasonal_matrix = np.array([digital_factors, traditional_factors, construction_factors])
im = ax2.imshow(seasonal_matrix, cmap='RdYlGn', vmin=0.7, vmax=1.3, aspect='auto')

ax2.set_xticks(np.arange(4))
ax2.set_yticks(np.arange(3))
ax2.set_xticklabels(['Q1', 'Q2', 'Q3', 'Q4'])
ax2.set_yticklabels(['Digital', 'Traditional', 'Construction'])

# Add text annotations
for i in range(3):
    for j in range(4):
        text = ax2.text(j, i, f'{seasonal_matrix[i, j]:.2f}',
                       ha="center", va="center", color="black", fontsize=10)

ax2.set_title('(b) Seasonal Factors Heatmap', fontsize=12, fontweight='bold')
ax2.set_xlabel('Quarter', fontsize=11)
plt.colorbar(im, ax=ax2, label='Seasonal Factor')

# Panel C: Coefficient of Variation by Quarter
ax3 = axes[1, 0]
digital_cv = [np.std([quarterly_digital.values[i] for i in range(16) if i % 4 == q]) /
              np.mean([quarterly_digital.values[i] for i in range(16) if i % 4 == q])
              for q in range(4)]
traditional_cv = [np.std([quarterly_traditional.values[i] for i in range(16) if i % 4 == q]) /
                  np.mean([quarterly_traditional.values[i] for i in range(16) if i % 4 == q])
                  for q in range(4)]
construction_cv = [np.std([quarterly_construction.values[i] for i in range(16) if i % 4 == q]) /
                   np.mean([quarterly_construction.values[i] for i in range(16) if i % 4 == q])
                   for q in range(4)]

ax3.plot(quarters_list, digital_cv, 'o-', label='Digital', color='#2E86AB', linewidth=2, markersize=8)
ax3.plot(quarters_list, traditional_cv, 's-', label='Traditional', color='#F18F01', linewidth=2, markersize=8)
ax3.plot(quarters_list, construction_cv, '^-', label='Construction', color='#6B8E23', linewidth=2, markersize=8)

ax3.set_title('(c) Quarterly Variation (CV)', fontsize=12, fontweight='bold')
ax3.set_xlabel('Quarter', fontsize=11)
ax3.set_ylabel('Coefficient of Variation', fontsize=11)
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

# Panel D: Box plot by quarter for Construction
ax4 = axes[1, 1]
construction_by_quarter = {f'Q{q+1}': [quarterly_construction.values[i]
                          for i in range(16) if i % 4 == q] for q in range(4)}
box_data = [construction_by_quarter[q] for q in ['Q1', 'Q2', 'Q3', 'Q4']]

bp = ax4.boxplot(box_data, labels=['Q1', 'Q2', 'Q3', 'Q4'], patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#6B8E23')
    patch.set_alpha(0.7)

ax4.set_title('(d) Construction Quarterly Distribution', fontsize=12, fontweight='bold')
ax4.set_xlabel('Quarter', fontsize=11)
ax4.set_ylabel('Number of AI Job Postings', fontsize=11)
ax4.grid(True, alpha=0.3, axis='y')

plt.suptitle('Seasonal Patterns in AI Job Postings', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figure3_seasonal_patterns.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Figure 3 saved as 'figure3_seasonal_patterns.png'")

# ============================================
# STATISTICAL TESTS FOR SEASONALITY
# ============================================

print("\n" + "=" * 60)
print("STATISTICAL TESTS FOR SEASONALITY")
print("=" * 60)

print("\n📊 KRUSKAL-WALLIS TEST (comparing quarters)")
print("-" * 40)

def kruskal_wallis_test(data):
    """Test if distributions differ across quarters"""
    q1 = [data[i] for i in range(len(data)) if i % 4 == 0]
    q2 = [data[i] for i in range(len(data)) if i % 4 == 1]
    q3 = [data[i] for i in range(len(data)) if i % 4 == 2]
    q4 = [data[i] for i in range(len(data)) if i % 4 == 3]

    statistic, p_value = stats.kruskal(q1, q2, q3, q4)
    return statistic, p_value

# Perform tests
digital_kw_stat, digital_kw_p = kruskal_wallis_test(quarterly_digital.values)
traditional_kw_stat, traditional_kw_p = kruskal_wallis_test(quarterly_traditional.values)
construction_kw_stat, construction_kw_p = kruskal_wallis_test(quarterly_construction.values)

print("H0: No difference in distribution across quarters")
print(f"\nDigital Sector:")
print(f"  H-statistic: {digital_kw_stat:.3f}, p-value: {digital_kw_p:.4f}")
print(f"  Result: {'Significant seasonality' if digital_kw_p < 0.05 else 'No significant seasonality'}")

print(f"\nTraditional Sector:")
print(f"  H-statistic: {traditional_kw_stat:.3f}, p-value: {traditional_kw_p:.4f}")
print(f"  Result: {'Significant seasonality' if traditional_kw_p < 0.05 else 'No significant seasonality'}")

print(f"\nConstruction Industry:")
print(f"  H-statistic: {construction_kw_stat:.3f}, p-value: {construction_kw_p:.4f}")
print(f"  Result: {'Significant seasonality' if construction_kw_p < 0.05 else 'No significant seasonality'}")

# ============================================
# TABLE 2: SEASONALITY SUMMARY
# ============================================

print("\n" + "=" * 60)
print("TABLE 2: SEASONALITY ANALYSIS SUMMARY")
print("=" * 60)

# Create summary table
seasonality_summary = pd.DataFrame({
    'Sector': ['Digital', 'Traditional', 'Construction'],
    'Seasonal Strength': [digital_strength, traditional_strength, construction_strength],
    'Peak Quarter': [
        f"Q{digital_factors.index(max(digital_factors)) + 1}",
        f"Q{traditional_factors.index(max(traditional_factors)) + 1}",
        f"Q{construction_factors.index(max(construction_factors)) + 1}"
    ],
    'Peak Factor': [max(digital_factors), max(traditional_factors), max(construction_factors)],
    'Trough Quarter': [
        f"Q{digital_factors.index(min(digital_factors)) + 1}",
        f"Q{traditional_factors.index(min(traditional_factors)) + 1}",
        f"Q{construction_factors.index(min(construction_factors)) + 1}"
    ],
    'Trough Factor': [min(digital_factors), min(traditional_factors), min(construction_factors)],
    'Range': [max(digital_factors) - min(digital_factors),
              max(traditional_factors) - min(traditional_factors),
              max(construction_factors) - min(construction_factors)],
    'KW p-value': [digital_kw_p, traditional_kw_p, construction_kw_p]
})

# Format for display
display_seasonality = seasonality_summary.copy()
display_seasonality['Seasonal Strength'] = display_seasonality['Seasonal Strength'].apply(lambda x: f'{x:.3f}')
display_seasonality['Peak Factor'] = display_seasonality['Peak Factor'].apply(lambda x: f'{x:.3f}')
display_seasonality['Trough Factor'] = display_seasonality['Trough Factor'].apply(lambda x: f'{x:.3f}')
display_seasonality['Range'] = display_seasonality['Range'].apply(lambda x: f'{x:.3f}')
display_seasonality['KW p-value'] = display_seasonality['KW p-value'].apply(lambda x: f'{x:.4f}')

print("\n", display_seasonality.to_string(index=False))

# Save table
seasonality_summary.to_csv('table2_seasonality_summary.csv', index=False)
print("\n✅ Table 2 saved as 'table2_seasonality_summary.csv'")

print("\n" + "=" * 60)
print("✅ SEASONALITY ANALYSIS COMPLETE")
print("=" * 60)
print("\nKey findings:")
print("1. Seasonal strength metrics calculated")
print("2. Seasonal decomposition completed")
print("3. Statistical tests performed")
print("4. Figures and tables generated")
print("\nPlease share these results before proceeding to the next analysis step.")


**Step 3.3: Cross-Industry Comparative Analysis and Lead-Lag Relationships**


In [ ]:
# ============================================
# COMPLETE DATA LOADING AND CROSS-INDUSTRY ANALYSIS
# ============================================
"""
Complete self-contained analysis including data loading,
preparation, and cross-industry comparative analysis
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import grangercausalitytests
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("COMPLETE ANALYSIS: DATA LOADING + CROSS-INDUSTRY COMPARISON")
print("=" * 60)

# ============================================
# STEP 1: DATA LOADING
# ============================================

print("\n📊 STEP 1: LOADING DATA FROM EXCEL")
print("-" * 40)

# Define the data path
DATA_PATH = 'YOUR_PATH'
FILE_NAME = 'all_industries_merged.xlsx'
FULL_PATH = DATA_PATH + FILE_NAME

try:
    # Load the Excel file
    print(f"Loading: {FILE_NAME}")
    df = pd.read_excel(FULL_PATH)
    print(f"✅ Data loaded: {len(df):,} records")

except Exception as e:
    print(f"❌ Error loading file: {str(e)}")
    raise

# ============================================
# STEP 2: DATA PREPARATION
# ============================================

print("\n📊 STEP 2: PREPARING DATA")
print("-" * 40)

# Parse dates if needed
if df['JobDate'].dtype == 'object':
    df['JobDate'] = pd.to_datetime(df['JobDate'])
elif pd.api.types.is_numeric_dtype(df['JobDate']):
    # Excel serial date conversion
    df['JobDate'] = pd.to_datetime('[redacted_phone]') + pd.to_timedelta(df['JobDate'], 'D')

# Extract temporal components
df['year'] = df['JobDate'].dt.year
df['month'] = df['JobDate'].dt.month
df['quarter'] = df['JobDate'].dt.quarter
df['year_quarter'] = df['JobDate'].dt.to_period('Q')

# Industry classification function
def classify_industry(anzsic_code):
    """Classify industry as Digital or Traditional"""
    if pd.isna(anzsic_code):
        return 'Unknown'

    code = int(anzsic_code)

    # Digital industries
    if 5800 <= code <= 6399:  # IT, Media, Finance
        return 'Digital'
    elif 6900 <= code <= 7299:  # Professional Services
        return 'Digital'
    # Traditional industries
    elif 3000 <= code <= 3999:  # Construction
        return 'Traditional'
    else:
        return 'Traditional' if code < 7300 else 'Other'

# Apply classifications
df['sector_type'] = df['ANZSIC CODE'].apply(classify_industry)
df['is_construction'] = df['ANZSIC CODE'].between(3000, 3999)

print(f"✅ Data prepared with sector classifications")
print(f"   Digital: {(df['sector_type'] == 'Digital').sum():,}")
print(f"   Traditional: {(df['sector_type'] == 'Traditional').sum():,}")
print(f"   Construction: {df['is_construction'].sum():,}")

# ============================================
# STEP 3: QUARTERLY AGGREGATION
# ============================================

print("\n📊 STEP 3: CREATING QUARTERLY TIME SERIES")
print("-" * 40)

# Filter for complete quarters (2021 Q1 - 2024 Q4)
df_analysis = df[df['year'] <= 2024].copy()

# Create quarterly aggregations
quarterly_digital = df_analysis[df_analysis['sector_type'] == 'Digital'].groupby('year_quarter').size()
quarterly_traditional = df_analysis[df_analysis['sector_type'] == 'Traditional'].groupby('year_quarter').size()
quarterly_construction = df_analysis[df_analysis['is_construction']].groupby('year_quarter').size()

# Ensure proper indexing
quarters = pd.period_range('2021Q1', '2024Q4', freq='Q')
quarterly_digital = quarterly_digital.reindex(quarters, fill_value=0)
quarterly_traditional = quarterly_traditional.reindex(quarters, fill_value=0)
quarterly_construction = quarterly_construction.reindex(quarters, fill_value=0)

print(f"✅ Created quarterly series for {len(quarters)} quarters")
print(f"   Digital total: {quarterly_digital.sum():,}")
print(f"   Traditional total: {quarterly_traditional.sum():,}")
print(f"   Construction total: {quarterly_construction.sum():,}")

# ============================================
# CROSS-CORRELATION ANALYSIS
# ============================================

print("\n" + "=" * 60)
print("CROSS-CORRELATION ANALYSIS")
print("=" * 60)

def calculate_cross_correlation(series1, series2, max_lag=4):
    """Calculate cross-correlation at different lags"""
    # Normalize series
    s1 = (series1 - np.mean(series1)) / (np.std(series1) + 1e-10)
    s2 = (series2 - np.mean(series2)) / (np.std(series2) + 1e-10)

    correlations = []
    lags = list(range(-max_lag, max_lag + 1))

    for lag in lags:
        if lag < 0:
            if len(s1[:lag]) > 0 and len(s2[-lag:]) > 0:
                corr = np.corrcoef(s1[:lag], s2[-lag:])[0, 1]
            else:
                corr = 0
        elif lag > 0:
            if len(s1[lag:]) > 0 and len(s2[:-lag]) > 0:
                corr = np.corrcoef(s1[lag:], s2[:-lag])[0, 1]
            else:
                corr = 0
        else:
            corr = np.corrcoef(s1, s2)[0, 1]

        correlations.append(corr if not np.isnan(corr) else 0)

    return lags, correlations

# Calculate cross-correlations
lags_dc, corr_dc = calculate_cross_correlation(quarterly_digital.values, quarterly_construction.values)
lags_tc, corr_tc = calculate_cross_correlation(quarterly_traditional.values, quarterly_construction.values)

# Find peak correlations
peak_dc_idx = np.argmax(np.abs(corr_dc))
peak_tc_idx = np.argmax(np.abs(corr_tc))

print("\n📈 Peak Cross-Correlations:")
print(f"Digital → Construction: Lag {lags_dc[peak_dc_idx]} quarters, r = {corr_dc[peak_dc_idx]:.3f}")
print(f"Traditional → Construction: Lag {lags_tc[peak_tc_idx]} quarters, r = {corr_tc[peak_tc_idx]:.3f}")

# ============================================
# GRANGER CAUSALITY TESTS
# ============================================

print("\n" + "=" * 60)
print("GRANGER CAUSALITY TESTS")
print("=" * 60)

def perform_granger_test(cause_series, effect_series, name1="Series1", name2="Series2"):
    """Perform Granger causality test"""
    data = pd.DataFrame({
        'cause': cause_series,
        'effect': effect_series
    })

    print(f"\nTesting: Does {name1} Granger-cause {name2}?")

    best_p = 1.0
    for lag in range(1, 4):
        try:
            result = grangercausalitytests(data[['effect', 'cause']], maxlag=[lag], verbose=False)
            p_value = result[lag][0]['ssr_ftest'][1]
            print(f"  Lag {lag}: p-value = {p_value:.4f}")
            best_p = min(best_p, p_value)
        except:
            pass

    if best_p < 0.05:
        print(f"  ✓ Significant Granger causality (p < 0.05)")
    elif best_p < 0.10:
        print(f"  ⚠ Marginal Granger causality (p < 0.10)")
    else:
        print(f"  ✗ No significant Granger causality")

    return best_p

# Perform tests
gc_dc_p = perform_granger_test(quarterly_digital.values, quarterly_construction.values,
                                "Digital", "Construction")
gc_tc_p = perform_granger_test(quarterly_traditional.values, quarterly_construction.values,
                                "Traditional", "Construction")

# ============================================
# CONVERGENCE/DIVERGENCE ANALYSIS
# ============================================

print("\n" + "=" * 60)
print("CONVERGENCE/DIVERGENCE ANALYSIS")
print("=" * 60)

# Calculate ratios
ratio_cd = quarterly_construction / (quarterly_digital + 1e-10)
ratio_ct = quarterly_construction / (quarterly_traditional + 1e-10)

# Fit linear trends
X = np.arange(len(ratio_cd)).reshape(-1, 1)

lr_cd = LinearRegression().fit(X, ratio_cd.values)
lr_ct = LinearRegression().fit(X, ratio_ct.values)

print(f"\nConstruction/Digital ratio trend: {lr_cd.coef_[0]:.4f}")
print(f"  → {'Converging' if lr_cd.coef_[0] > 0 else 'Diverging'}")

print(f"\nConstruction/Traditional ratio trend: {lr_ct.coef_[0]:.4f}")
print(f"  → {'Converging' if lr_ct.coef_[0] > 0 else 'Diverging'}")

# Calculate gaps
avg_gap_digital = (quarterly_digital.mean() - quarterly_construction.mean()) / quarterly_construction.mean() * 100
avg_gap_traditional = (quarterly_traditional.mean() - quarterly_construction.mean()) / quarterly_construction.mean() * 100

print(f"\nAverage Gaps ([redacted_phone]):")
print(f"  Digital has {avg_gap_digital:.0f}% more AI jobs than Construction")
print(f"  Traditional has {avg_gap_traditional:.0f}% more AI jobs than Construction")

# ============================================
# FIGURE: COMPARATIVE ANALYSIS
# ============================================

print("\n" + "=" * 60)
print("CREATING COMPARATIVE ANALYSIS FIGURE")
print("=" * 60)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel A: Cross-correlations
ax1 = axes[0, 0]
width = 0.35
x_pos = np.array(lags_dc)
ax1.bar(x_pos - width/2, corr_dc, width, color='#2E86AB', alpha=0.7, label='Digital-Construction')
ax1.bar(x_pos + width/2, corr_tc, width, color='#F18F01', alpha=0.7, label='Traditional-Construction')
ax1.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
ax1.axvline(x=0, color='gray', linestyle='--', linewidth=0.5)
ax1.set_title('(a) Cross-Correlation with Construction', fontsize=12, fontweight='bold')
ax1.set_xlabel('Lag (quarters)', fontsize=11)
ax1.set_ylabel('Correlation', fontsize=11)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Panel B: Normalized trajectories
ax2 = axes[0, 1]
dates = pd.to_datetime(quarters.to_timestamp())
ax2.plot(dates, quarterly_digital.values/quarterly_digital.iloc[0]*100,
         '-o', color='#2E86AB', linewidth=2, label='Digital', markersize=4)
ax2.plot(dates, quarterly_traditional.values/quarterly_traditional.iloc[0]*100,
         '-s', color='#F18F01', linewidth=2, label='Traditional', markersize=4)
ax2.plot(dates, quarterly_construction.values/quarterly_construction.iloc[0]*100,
         '-^', color='#6B8E23', linewidth=2, label='Construction', markersize=4)
ax2.set_title('(b) Growth Trajectories (2021Q1=100)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Quarter', fontsize=11)
ax2.set_ylabel('Index', fontsize=11)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.axhline(y=100, color='gray', linestyle='--', alpha=0.5)

# Format x-axis
ax2.set_xticks(dates[::2])
labels = [f'{quarters[i].year}Q{quarters[i].quarter}' for i in range(0, len(quarters), 2)]
ax2.set_xticklabels(labels, rotation=45, ha='right')

# Panel C: Ratios
ax3 = axes[1, 0]
ax3.plot(dates, ratio_cd.values, '-o', color='#2E86AB', linewidth=2,
         label='Construction/Digital', markersize=5)
ax3.plot(dates, ratio_ct.values, '-s', color='#F18F01', linewidth=2,
         label='Construction/Traditional', markersize=5)
ax3.plot(dates, lr_cd.predict(X), '--', color='#2E86AB', alpha=0.5)
ax3.plot(dates, lr_ct.predict(X), '--', color='#F18F01', alpha=0.5)
ax3.set_title('(c) Relative Adoption Ratios', fontsize=12, fontweight='bold')
ax3.set_xlabel('Quarter', fontsize=11)
ax3.set_ylabel('Ratio', fontsize=11)
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.set_xticks(dates[::2])
ax3.set_xticklabels(labels, rotation=45, ha='right')

# Panel D: Gaps
ax4 = axes[1, 1]
gap_digital = ((quarterly_digital - quarterly_construction) / (quarterly_construction + 1e-10) * 100)
gap_traditional = ((quarterly_traditional - quarterly_construction) / (quarterly_construction + 1e-10) * 100)
gap_digital = gap_digital.replace([np.inf, -np.inf], 0)
gap_traditional = gap_traditional.replace([np.inf, -np.inf], 0)

ax4.fill_between(dates, 0, gap_digital.values, color='#2E86AB', alpha=0.3, label='Digital Gap')
ax4.fill_between(dates, 0, gap_traditional.values, color='#F18F01', alpha=0.3, label='Traditional Gap')
ax4.plot(dates, gap_digital.values, '-', color='#2E86AB', linewidth=2)
ax4.plot(dates, gap_traditional.values, '-', color='#F18F01', linewidth=2)
ax4.set_title('(d) Adoption Gap (% above Construction)', fontsize=12, fontweight='bold')
ax4.set_xlabel('Quarter', fontsize=11)
ax4.set_ylabel('Gap (%)', fontsize=11)
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)
ax4.set_xticks(dates[::2])
ax4.set_xticklabels(labels, rotation=45, ha='right')

plt.suptitle('Cross-Industry Comparative Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('cross_industry_analysis_complete.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Analysis complete and figure saved!")

# ============================================
# SUMMARY TABLE
# ============================================

print("\n" + "=" * 60)
print("SUMMARY OF KEY FINDINGS")
print("=" * 60)

summary = pd.DataFrame({
    'Metric': [
        'Peak Cross-Correlation (Digital→Construction)',
        'Peak Cross-Correlation (Traditional→Construction)',
        'Granger Causality p-value (Digital→Construction)',
        'Granger Causality p-value (Traditional→Construction)',
        'Average Gap: Digital vs Construction (%)',
        'Average Gap: Traditional vs Construction (%)',
        'Convergence Trend (Construction/Digital)',
        'Convergence Trend (Construction/Traditional)'
    ],
    'Value': [
        f"{corr_dc[peak_dc_idx]:.3f} at lag {lags_dc[peak_dc_idx]}",
        f"{corr_tc[peak_tc_idx]:.3f} at lag {lags_tc[peak_tc_idx]}",
        f"{gc_dc_p:.4f}",
        f"{gc_tc_p:.4f}",
        f"{avg_gap_digital:.0f}%",
        f"{avg_gap_traditional:.0f}%",
        'Converging' if lr_cd.coef_[0] > 0 else 'Diverging',
        'Converging' if lr_ct.coef_[0] > 0 else 'Diverging'
    ]
})

print("\n", summary.to_string(index=False))

print("\n" + "=" * 60)
print("✅ COMPLETE ANALYSIS FINISHED")
print("=" * 60)


In [ ]:
# ============================================
# CREATE TABLE 3: CROSS-INDUSTRY COMPARATIVE SUMMARY
# ============================================
"""
Generate Table 3 based on the analysis results from Step 3.3
"""

import pandas as pd
import numpy as np

print("=" * 60)
print("CREATING TABLE 3: CROSS-INDUSTRY COMPARATIVE SUMMARY")
print("=" * 60)

# Based on the results from your analysis output:
# Peak Cross-Correlation (Digital→Construction): 0.382 at lag 0
# Peak Cross-Correlation (Traditional→Construction): -0.336 at lag 4
# Granger Causality p-value (Digital→Construction): 0.1109
# Granger Causality p-value (Traditional→Construction): 0.7894
# Average Gap: Digital vs Construction: 1710%
# Average Gap: Traditional vs Construction: 911%
# Convergence Trend (Construction/Digital): Diverging (-0.0018)
# Convergence Trend (Construction/Traditional): Diverging (-0.0041)

# Create the comprehensive summary table
table3 = pd.DataFrame({
    'Relationship': [
        'Digital → Construction',
        'Traditional → Construction',
        'Construction vs Digital Gap',
        'Construction vs Traditional Gap'
    ],
    'Peak Correlation': [
        0.382,
        -0.336,
        np.nan,
        np.nan
    ],
    'Optimal Lag (quarters)': [
        0,
        4,
        np.nan,
        np.nan
    ],
    'Granger Causality p-value': [
        0.1109,
        0.7894,
        np.nan,
        np.nan
    ],
    'Significant (p<0.05)': [
        'No',
        'No',
        np.nan,
        np.nan
    ],
    'Average Gap (%)': [
        np.nan,
        np.nan,
        1710,
        911
    ],
    'Trend': [
        'Contemporaneous',
        'Construction lags 4Q',
        'Diverging (-0.18%/Q)',
        'Diverging (-0.41%/Q)'
    ]
})

# Display version with better formatting
print("\nTABLE 3: Cross-Industry Comparative Analysis Summary")
print("-" * 60)

# Create a cleaner display version
display_table = table3.copy()

# Format numerical columns
display_table['Peak Correlation'] = display_table['Peak Correlation'].apply(
    lambda x: f'{x:.3f}' if not pd.isna(x) else '-'
)
display_table['Optimal Lag (quarters)'] = display_table['Optimal Lag (quarters)'].apply(
    lambda x: f'{int(x)}' if not pd.isna(x) else '-'
)
display_table['Granger Causality p-value'] = display_table['Granger Causality p-value'].apply(
    lambda x: f'{x:.4f}' if not pd.isna(x) else '-'
)
display_table['Significant (p<0.05)'] = display_table['Significant (p<0.05)'].apply(
    lambda x: x if not pd.isna(x) else '-'
)
display_table['Average Gap (%)'] = display_table['Average Gap (%)'].apply(
    lambda x: f'{int(x):,}%' if not pd.isna(x) else '-'
)

print(display_table.to_string(index=False))

# Save to CSV
table3.to_csv('table3_cross_industry_comparative.csv', index=False)
print("\n✅ Table 3 saved as 'table3_cross_industry_comparative.csv'")

# Create an alternative more detailed version
print("\n" + "=" * 60)
print("ALTERNATIVE TABLE 3: Detailed Comparative Metrics")
print("-" * 60)

# More detailed breakdown
detailed_table3 = pd.DataFrame({
    'Metric': [
        'Cross-Correlation: Digital→Construction',
        'Cross-Correlation: Traditional→Construction',
        'Granger Causality: Digital→Construction',
        'Granger Causality: Traditional→Construction',
        'Average Jobs/Quarter: Digital',
        'Average Jobs/Quarter: Traditional',
        'Average Jobs/Quarter: Construction',
        'Gap: Digital vs Construction',
        'Gap: Traditional vs Construction',
        'Divergence Rate: From Digital',
        'Divergence Rate: From Traditional',
        'CAGR: Digital',
        'CAGR: Traditional',
        'CAGR: Construction'
    ],
    'Value': [
        '0.382',
        '-0.336',
        'p = 0.111',
        'p = 0.789',
        '1,089',
        '608',
        '60',
        '1,710%',
        '911%',
        '-0.18% per quarter',
        '-0.41% per quarter',
        '+16.4%',
        '+11.8%',
        '+2.5%'
    ],
    'Interpretation': [
        'Weak positive, contemporaneous',
        'Negative correlation at 4Q lag',
        'No predictive power',
        'No predictive power',
        '18x more than construction',
        '10x more than construction',
        'Minimal AI adoption',
        'Massive gap',
        'Substantial gap',
        'Gap widening',
        'Gap widening',
        'Strong growth',
        'Moderate growth',
        'Stagnant'
    ]
})

print(detailed_table3.to_string(index=False))

# Save detailed version
detailed_table3.to_csv('table3_detailed_comparative.csv', index=False)
print("\n✅ Detailed Table 3 saved as 'table3_detailed_comparative.csv'")

print("\n" + "=" * 60)
print("KEY INSIGHTS FROM TABLE 3")
print("=" * 60)

print("""
1. CORRELATION PATTERNS:
   • Digital and Construction move together (r=0.382, lag 0)
   • Traditional and Construction show negative correlation (r=-0.336, lag 4)
   • Suggests construction follows different dynamics than both sectors

2. PREDICTIVE RELATIONSHIPS:
   • No significant Granger causality detected
   • Construction's AI adoption appears independent of other sectors
   • No evidence of technology spillover effects

3. ADOPTION GAPS:
   • Digital sector: 17-fold advantage (1,710%)
   • Traditional sector: 9-fold advantage (911%)
   • Construction severely underrepresented in AI workforce

4. DIVERGENCE TRENDS:
   • Gap with Digital widening at 0.18% per quarter
   • Gap with Traditional widening at 0.41% per quarter
   • No convergence observed - construction falling further behind

5. GROWTH DISPARITIES:
   • Digital growing 6.6x faster than Construction
   • Traditional growing 4.7x faster than Construction
   • Construction essentially stagnant in AI adoption
""")

print("\n✅ Table 3 creation complete!")
print("\nYou now have both a standard and detailed version of Table 3 for your paper.")


# Version 02


In [ ]:
# ============================================================
# AI Job Postings: Core Time-Series Analysis (Colab-ready)
# ============================================================
# What it does:
# - Reads all_industries_merged.xlsx
# - Maps ANZSIC -> sectors: Construction / Digital / Traditional
# - Builds monthly + quarterly series
# - Saves figures (PNG, 300dpi) and tables (CSV) with journal-ready filenames
# - Robust cross-correlation plotter (handles Matplotlib versions)
# ============================================================

# --- Optional installs for a fresh Colab (uncomment if needed) ---
# !pip install -q pandas numpy matplotlib statsmodels openpyxl

import os
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Try to import statsmodels for decomposition & ETS; script works even if unavailable.
STATS_OK = True
try:
    from statsmodels.tsa.seasonal import seasonal_decompose
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
except Exception:
    STATS_OK = False

# -------------------- CONFIG --------------------
SOURCE_XLSX = 'path/to/your_data.xlsx'
OUTPUT_DIR  = "YOUR_PATH"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Toggle the (optional) rolling ETS backtest:
DO_FORECAST = True          # set to False to skip
ROLLING_TEST_MONTHS = 12    # last N months for rolling one-step forecasts

# -------------------- LOAD ----------------------
xlsx = pd.ExcelFile(SOURCE_XLSX)
# Heuristic: pick a sheet called "data"/"all"/"merged"/"postings" if present, else first
prefer = next((s for s in xlsx.sheet_names if s.lower() in {"data","all","merged","postings"}),
              xlsx.sheet_names[0])
df = pd.read_excel(SOURCE_XLSX, sheet_name=prefer)

# Identify date & industry columns
date_col = None
for c in df.columns:
    if "date" in c.lower():
        date_col = c; break
if date_col is None:
    for c in df.columns:  # fallback: first datetime-like column
        try:
            pd.to_datetime(df[c])
            date_col = c; break
        except Exception:
            pass
assert date_col is not None, "Could not find a date column."

ind_col = None
for c in df.columns:
    cl = c.lower()
    if "anzsic" in cl or "industry" in cl or "sector" in cl:
        ind_col = c; break
assert ind_col is not None, "Could not find an industry/ANZSIC column."

df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
df = df.dropna(subset=[date_col]).sort_values(date_col)

# -------------------- CLASSIFY SECTORS ----------------------
def to_anzsic_numeric(val):
    if pd.isna(val): return np.nan
    try:
        return int(val)
    except Exception:
        m = re.search(r"\d+", str(val))
        return int(m.group()) if m else np.nan

anzsic_num = df[ind_col].apply(to_anzsic_numeric)

def sector_from_anzsic(code):
    if pd.isna(code): return "Unknown"
    code = int(code)
    # Construction: [redacted_phone]
    if 3000 <= code <= 3999:
        return "Construction"
    # Digital: [redacted_phone] and [redacted_phone] (per paper)
    if (5800 <= code <= 6399) or (6900 <= code <= 7299):
        return "Digital"
    return "Traditional"

df["sector_group"] = anzsic_num.apply(sector_from_anzsic)

# -------------------- BUILD SERIES ----------------------
df["month_ts"]   = df[date_col].dt.to_period("M").dt.to_timestamp()
df["quarter_ts"] = df[date_col].dt.to_period("Q").dt.to_timestamp()

m_counts = (
    df.groupby(["month_ts","sector_group"]).size()
      .rename("count").reset_index()
)
q_counts = (
    df.groupby(["quarter_ts","sector_group"]).size()
      .rename("count").reset_index()
)

m_pivot = (m_counts.pivot(index="month_ts", columns="sector_group", values="count")
           .fillna(0).sort_index())
q_pivot = (q_counts.pivot(index="quarter_ts", columns="sector_group", values="count")
           .fillna(0).sort_index())

# -------------------- UTILITIES ----------------------
def save_line_plot(df_ts, title, ylabel, fname):
    plt.figure()
    for col in df_ts.columns:
        if pd.api.types.is_numeric_dtype(df_ts[col]):
            plt.plot(df_ts.index, df_ts[col], label=col)
    plt.title(title)
    plt.xlabel("Time"); plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, fname)
    plt.savefig(path, dpi=300)
    plt.close()
    return path

def growth_index(series):
    s = series.dropna()
    if len(s)==0:
        return series*math.nan
    base = s.iloc[0] if s.iloc[0] != 0 else (s[s!=0].iloc[0] if (s!=0).any() else 1.0)
    return (series / base) * 100.0

def safe_ccf(x, y, max_lag=8):
    """Return lags (-max..+max) and correlations; +lag means y leads x."""
    x = pd.Series(x, dtype=float); y = pd.Series(y, dtype=float)
    x = (x - x.mean()) / (x.std(ddof=0) or 1)
    y = (y - y.mean()) / (y.std(ddof=0) or 1)
    lags = np.arange(-max_lag, max_lag+1)
    corrs = []
    for L in lags:
        if L < 0:
            xs, ys = x[-L:], y[:len(y)+L]
        elif L > 0:
            xs, ys = x[:len(x)-L], y[L:]
        else:
            xs, ys = x, y
        corr = np.corrcoef(xs.values, ys.values)[0,1] if len(xs) > 3 else np.nan
        corrs.append(corr)
    return lags, np.array(corrs)

def basic_stats(ts):
    ts = pd.Series(ts).dropna()
    return pd.Series({
        "count": ts.size,
        "mean": ts.mean(),
        "std": ts.std(ddof=1),
        "min": ts.min(),
        "max": ts.max(),
        "coef_variation": (ts.std(ddof=1)/ts.mean()) if ts.mean()!=0 else np.nan
    })

# Robust CCF plotter (handles Matplotlib versions)
def plot_ccf_stem_or_bar(lags, corrs, title, outpath):
    plt.figure()
    try:
        # Newer Matplotlib: no 'use_line_collection' arg; plain call works
        plt.stem(lags, corrs)
    except TypeError:
        # Fallback: simple bar chart if stem signature fails
        plt.bar(lags, corrs, width=0.8, align="center")
    plt.axhline(0, linewidth=1)
    plt.title(title)
    plt.xlabel("Lag (months)")
    plt.ylabel("Correlation")
    plt.tight_layout()
    plt.savefig(outpath, dpi=300)
    plt.close()

# -------------------- FIGURES: TRENDS ----------------------
fig1 = save_line_plot(q_pivot[["Construction","Digital","Traditional"]].fillna(0),
                      "Quarterly AI Job Postings by Sector",
                      "Job postings",
                      "Fig01_Quarterly_Postings.png")

qi = q_pivot.apply(growth_index)
fig2 = save_line_plot(qi[["Construction","Digital","Traditional"]].fillna(0),
                      "Growth Index by Sector (Quarterly, base=first observation)",
                      "Index (Q1=100)",
                      "Fig02_GrowthIndex_Quarterly.png")

# -------------------- SEASONALITY ----------------------
seasonality_rows = []
seasonal_figs = {}

if STATS_OK and len(m_pivot) >= 24:
    # STL additive decomposition per sector (period=12)
    for sec in ["Construction","Digital","Traditional"]:
        if sec in m_pivot.columns:
            series = m_pivot[sec].asfreq("MS").fillna(0)
            try:
                dec = seasonal_decompose(series, model="additive", period=12, extrapolate_trend="freq")
                S, R = dec.seasonal, dec.resid
                valid = (~S.isna()) & (~R.isna())
                var_R = np.var(R[valid]); var_SR = np.var((S[valid] + R[valid]))
                fs = max(0.0, 1.0 - (var_R/var_SR)) if var_SR>0 else np.nan
                # Save plot
                fig = dec.plot()
                plt.suptitle(f"Seasonal Decomposition ({sec})", y=1.02)
                plt.tight_layout()
                pth = os.path.join(OUTPUT_DIR, f"Fig0{3 + ['Construction','Digital','Traditional'].index(sec)}_STL_{sec}.png")
                plt.savefig(pth, dpi=300)
                plt.close()
                seasonal_figs[sec] = pth
                seasonality_rows.append({"sector": sec, "seasonal_strength": fs})
            except Exception:
                seasonality_rows.append({"sector": sec, "seasonal_strength": np.nan})
else:
    # Proxy: variance explained by month-of-year
    tmp = m_pivot.copy()
    tmp["m"] = tmp.index.month
    for sec in ["Construction","Digital","Traditional"]:
        if sec in tmp.columns:
            overall = tmp[sec].mean()
            mm = tmp.groupby("m")[sec].mean()
            ss_between = ((mm - overall)**2).sum()
            ss_total = ((tmp[sec] - overall)**2).sum()
            fs = (ss_between / ss_total) if ss_total>0 else np.nan
            seasonality_rows.append({"sector": sec, "seasonal_strength_proxy": fs})

# Quarter profile (relative level by calendar quarter: peak/trough)
q_tmp = q_pivot.copy(); q_tmp["qnum"] = q_tmp.index.quarter
peak_rows = []
for sec in ["Construction","Digital","Traditional"]:
    if sec in q_tmp.columns:
        overall = q_tmp[sec].mean()
        prof = q_tmp.groupby("qnum")[sec].mean() / (overall if overall>0 else 1)
        peak_q   = int(prof.idxmax()) if len(prof)>0 else np.nan
        trough_q = int(prof.idxmin()) if len(prof)>0 else np.nan
        peak_rows.append({"sector": sec, "peak_quarter": peak_q, "trough_quarter": trough_q})

seasonality_df = pd.DataFrame(seasonality_rows)
peaks_df       = pd.DataFrame(peak_rows)

# Save Table 3
tbl3 = pd.merge(seasonality_df, peaks_df, on="sector", how="outer")
tbl3_path = os.path.join(OUTPUT_DIR, "Table03_SeasonalitySummary.csv")
tbl3.to_csv(tbl3_path, index=False)

# -------------------- CROSS-CORRELATION ----------------------
cc_rows = []
if "Construction" in m_pivot.columns:
    cons = m_pivot["Construction"].astype(float)
    for other in ["Digital","Traditional"]:
        if other in m_pivot.columns:
            lags, corrs = safe_ccf(cons.values, m_pivot[other].astype(float).values, max_lag=8)

            title   = f"Cross-correlation: Construction vs {other}\n(+lag: {other} leads)"
            outname = f"Fig0{6 if other=='Digital' else 7}_CCF_Cons_{other}.png"
            outpath = os.path.join(OUTPUT_DIR, outname)
            plot_ccf_stem_or_bar(lags, corrs, title, outpath)

            # Peak absolute correlation
            i = int(np.nanargmax(np.abs(corrs)))
            peak_r, peak_lag = float(corrs[i]), int(lags[i])
            cc_rows.append({"pair": f"Construction vs {other}",
                            "peak_corr": peak_r,
                            "lag_at_peak_months": peak_lag})

tbl4 = pd.DataFrame(cc_rows)
tbl4_path = os.path.join(OUTPUT_DIR, "Table04_CrossCorrelation.csv")
tbl4.to_csv(tbl4_path, index=False)

# -------------------- CONVERGENCE / DIVERGENCE (RATIOS) ----------------------
ratio_rows = []
if "Construction" in q_pivot.columns:
    cons_q = q_pivot["Construction"].replace(0, np.nan)
    for other in ["Digital","Traditional"]:
        if other in q_pivot.columns:
            oth_q = q_pivot[other].replace(0, np.nan)
            ratio = cons_q / oth_q
            x = np.arange(len(ratio)); y = ratio.values
            mask = ~np.isnan(y)
            slope = np.polyfit(x[mask], y[mask], 1)[0] if mask.sum()>=4 else np.nan

            # plot
            plt.figure()
            plt.plot(q_pivot.index, ratio, marker="o")
            plt.title(f"Ratio over time: Construction / {other}")
            plt.xlabel("Quarter"); plt.ylabel("Ratio")
            plt.tight_layout()
            outname = f"Fig0{8 if other=='Digital' else 9}_Ratio_Cons_{other}.png"
            outpath = os.path.join(OUTPUT_DIR, outname)
            plt.savefig(outpath, dpi=300); plt.close()

            ratio_rows.append({
                "ratio": f"Construction/{other}",
                "slope_per_quarter": float(slope) if slope==slope else np.nan,
                "start_ratio": float(pd.Series(ratio).dropna().iloc[0]) if pd.Series(ratio).dropna().size>0 else np.nan,
                "end_ratio": float(pd.Series(ratio).dropna().iloc[-1]) if pd.Series(ratio).dropna().size>0 else np.nan
            })

tbl5 = pd.DataFrame(ratio_rows)
tbl5_path = os.path.join(OUTPUT_DIR, "Table05_RatioTrends.csv")
tbl5.to_csv(tbl5_path, index=False)

# -------------------- TABLE 1 & 2 + coverage ----------------------
totals = df.groupby("sector_group").size().rename("n_postings").reset_index()
min_date, max_date = df[date_col].min(), df[date_col].max()
coverage = pd.DataFrame([{
    "min_date": min_date, "max_date": max_date,
    "n_months": len(m_pivot), "n_quarters": len(q_pivot)
}])
tbl1_coverage_path = os.path.join(OUTPUT_DIR, "Table01_TotalsAndCoverage.csv")
pd.concat([totals, coverage], axis=1).to_csv(tbl1_coverage_path, index=False)

diag = q_pivot.apply(basic_stats).T.reset_index().rename(columns={"index":"sector"})
tbl2_diag_path = os.path.join(OUTPUT_DIR, "Table02_QuarterlyDiagnostics.csv")
diag.to_csv(tbl2_diag_path, index=False)

# -------------------- OPTIONAL: ROLLING ETS BACKTEST ----------------------
tbl6_path = None
if DO_FORECAST and STATS_OK:
    rows = []
    # Do on monthly series (finer granularity)
    for sec in ["Construction","Digital","Traditional"]:
        if sec in m_pivot.columns:
            y = m_pivot[sec].astype(float).copy()
            y = y.asfreq("MS").fillna(0)
            if len(y) >= (ROLLING_TEST_MONTHS + 24):  # ensure enough history
                errs_abs = []
                errs_pct = []
                for t in range(ROLLING_TEST_MONTHS, 0, -1):
                    train = y.iloc[: -t]
                    true_next = y.iloc[-t]
                    try:
                        model = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=12)
                        fit = model.fit(optimized=True, use_brute=True)
                        pred = fit.forecast(1).iloc[0]
                        e = abs(pred - true_next)
                        errs_abs.append(e)
                        denom = true_next if true_next != 0 else 1.0
                        errs_pct.append(e / denom)
                    except Exception:
                        pass
                if errs_abs:
                    mae  = float(np.mean(errs_abs))
                    mape = float(np.mean(errs_pct))
                else:
                    mae, mape = np.nan, np.nan
                rows.append({"sector": sec, "rolling_months": ROLLING_TEST_MONTHS,
                             "MAE": mae, "MAPE": mape})
    if rows:
        tbl6 = pd.DataFrame(rows)
        tbl6_path = os.path.join(OUTPUT_DIR, "Table06_RollingETS_ForecastAccuracy.csv")
        tbl6.to_csv(tbl6_path, index=False)

# -------------------- PRINT SUMMARY ----------------------
print("Saved figures:")
print("  Fig01_Quarterly_Postings.png")
print("  Fig02_GrowthIndex_Quarterly.png")
print("  Fig03_STL_Construction.png (if statsmodels present)")
print("  Fig04_STL_Digital.png (if statsmodels present)")
print("  Fig05_STL_Traditional.png (if statsmodels present)")
print("  Fig06_CCF_Cons_Digital.png")
print("  Fig07_CCF_Cons_Traditional.png")
print("  Fig08_Ratio_Cons_Digital.png")
print("  Fig09_Ratio_Cons_Traditional.png")
print("\nSaved tables:")
print("  Table01_TotalsAndCoverage.csv")
print("  Table02_QuarterlyDiagnostics.csv")
print("  Table03_SeasonalitySummary.csv")
print("  Table04_CrossCorrelation.csv")
print("  Table05_RatioTrends.csv")
if tbl6_path:
    print("  Table06_RollingETS_ForecastAccuracy.csv")
print(f"\nOutput folder: {OUTPUT_DIR}")


# Version 03


In [ ]:
# ============================================================
# AI Job Postings: Time-Series Analysis (Colab-ready, Stable v3)
# ============================================================
# What this does:
# - Rebuilds all previous outputs (Fig01–Fig09, Table01–Table06)
# - Adds: Kruskal–Wallis seasonality tests (Table07)
#         ARIMA vs ETS comparison + rolling 1-step forecasts (Fig10–12, Table08)
#         Bootstrap CCF with CI bands (Fig13–14, Table09 + per-lag CI CSVs)
# - Stability: write locally first, then copy once to Drive at end.
# - Quality: PNG at 600 dpi; single-plot figures; Matplotlib only, no seaborn.
# ============================================================

# --- Optional installs for fresh Colab sessions ---
# !pip install -q pandas numpy matplotlib statsmodels openpyxl scipy

import os, re, math, warnings, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# -------------------- PATHS --------------------
SOURCE_XLSX      = 'path/to/your_data.xlsx'
OUTPUT_DIR_LOCAL = "YOUR_PATH"  # fast local disk
OUTPUT_DIR_DRIVE = "YOUR_PATH"

os.makedirs(OUTPUT_DIR_LOCAL, exist_ok=True)

# -------------------- TOGGLES --------------------
DO_CORE_FIGS      = True   # Fig01–Fig02 + Tables 01–02,05
DO_STL            = True   # Fig03–Fig05 + Table03
DO_CCF_SIMPLE     = True   # Fig06–Fig07 + Table04
DO_RATIOS         = True   # Fig08–Fig09 + Table05 (already saved in CORE if you prefer)
DO_ETS_ROLLING    = True   # Table06 (rolling 1-step ETS)
DO_KRUSKAL        = True   # Table07 (seasonality test across quarters)
DO_ARIMA_VS_ETS   = True   # Fig10–Fig12 + Table08
DO_BOOTSTRAP_CCF  = True   # Fig13–Fig14 + Table09 (+ per-lag CI CSVs)

# -------------------- LIGHTER SETTINGS (tweakable) --------------------
ROLLING_TEST_MONTHS = 8     # was 12
MAX_LAG             = 6     # months for CCF
N_BOOT              = 1500   # bootstrap reps (increase later if needed)
BLOCK_LEN           = 6     # months for moving block bootstrap
ARIMA_MAX_P         = 1     # AR/MA small grid
ARIMA_MAX_Q         = 1
SEAS_PERIODS        = 12    # monthly seasonality

# -------------------- MATPLOTLIB (journal-friendly) --------------------
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.figsize": (7, 4.2),
})

# -------------------- OPTIONAL IMPORTS --------------------
HAVE_STATS = True
HAVE_SCIPY = True
try:
    from statsmodels.tsa.seasonal import seasonal_decompose
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    from statsmodels.tsa.statespace.sarimax import SARIMAX
except Exception:
    HAVE_STATS = False

try:
    from scipy.stats import kruskal
except Exception:
    HAVE_SCIPY = False

# -------------------- LOAD DATA --------------------
xlsx = pd.ExcelFile(SOURCE_XLSX)
prefer = next((s for s in xlsx.sheet_names if s.lower() in {"data","all","merged","postings"}), xlsx.sheet_names[0])
df = pd.read_excel(SOURCE_XLSX, sheet_name=prefer)

# Identify date & industry columns
date_col = None
for c in df.columns:
    if "date" in c.lower():
        date_col = c; break
if date_col is None:
    for c in df.columns:
        try:
            pd.to_datetime(df[c])
            date_col = c; break
        except Exception:
            pass
assert date_col is not None, "Could not find a date column."

ind_col = None
for c in df.columns:
    cl = c.lower()
    if "anzsic" in cl or "industry" in cl or "sector" in cl:
        ind_col = c; break
assert ind_col is not None, "Could not find an industry/ANZSIC column."

df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
df = df.dropna(subset=[date_col]).sort_values(date_col)

# -------------------- CLASSIFY SECTORS --------------------
def to_anzsic_numeric(val):
    if pd.isna(val): return np.nan
    try: return int(val)
    except Exception:
        m = re.search(r"\d+", str(val))
        return int(m.group()) if m else np.nan

anzsic_num = df[ind_col].apply(to_anzsic_numeric)

def sector_from_anzsic(code):
    if pd.isna(code): return "Unknown"
    code = int(code)
    if 3000 <= code <= 3999: return "Construction"
    if (5800 <= code <= 6399) or (6900 <= code <= 7299): return "Digital"
    return "Traditional"

df["sector_group"] = anzsic_num.apply(sector_from_anzsic)

# -------------------- TIME INDEXES & PIVOTS --------------------
df["month_ts"]   = df[date_col].dt.to_period("M").dt.to_timestamp()
df["quarter_ts"] = df[date_col].dt.to_period("Q").dt.to_timestamp()

m_counts = df.groupby(["month_ts","sector_group"]).size().rename("count").reset_index()
q_counts = df.groupby(["quarter_ts","sector_group"]).size().rename("count").reset_index()

m_pivot = m_counts.pivot(index="month_ts", columns="sector_group", values="count").fillna(0).sort_index()
q_pivot = q_counts.pivot(index="quarter_ts", columns="sector_group", values="count").fillna(0).sort_index()

# -------------------- HELPERS --------------------
def save_line_plot(df_ts, title, ylabel, fname):
    plt.figure()
    for col in df_ts.columns:
        if pd.api.types.is_numeric_dtype(df_ts[col]):
            plt.plot(df_ts.index, df_ts[col], label=col)
    plt.title(title)
    plt.xlabel("Time"); plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR_LOCAL, fname)
    plt.savefig(path, dpi=600)
    plt.close()
    return path

def growth_index(series):
    s = series.dropna()
    if len(s)==0: return series*np.nan
    base = s.iloc[0] if s.iloc[0] != 0 else (s[s!=0].iloc[0] if (s!=0).any() else 1.0)
    return (series / base) * 100.0

def safe_ccf(x, y, max_lag=6):
    """
    Cross-correlation between x (Construction) and y (Comparator).
    Convention: positive lag (+L) indicates comparator y leads Construction x,
    matching manuscript Fig 7-8 captions ('positive lags indicate the comparator leads').
    """
    x = pd.Series(x, dtype=float); y = pd.Series(y, dtype=float)
    x = (x - x.mean()) / (x.std(ddof=0) or 1)
    y = (y - y.mean()) / (y.std(ddof=0) or 1)
    lags = np.arange(-max_lag, max_lag+1)
    corrs = []
    for L in lags:
        if L > 0:
            # Comparator y leads Construction x: corr(x_{t+L}, y_t)
            xs, ys = x[L:], y[:len(y)-L]
        elif L < 0:
            # Construction x leads comparator y: corr(x_t, y_{t+|L|})
            xs, ys = x[:len(x)+L], y[-L:]
        else:
            xs, ys = x, y
        corr = np.corrcoef(xs.values, ys.values)[0,1] if len(xs) > 3 else np.nan
        corrs.append(corr)
    return lags, np.array(corrs)

def basic_stats(ts):
    ts = pd.Series(ts).dropna()
    return pd.Series({
        "count": ts.size,
        "mean": ts.mean(),
        "std": ts.std(ddof=1),
        "min": ts.min(),
        "max": ts.max(),
        "coef_variation": (ts.std(ddof=1)/ts.mean()) if ts.mean()!=0 else np.nan
    })

def plot_ccf_stem_or_bar(lags, corrs, title, fname):
    plt.figure()
    try:
        plt.stem(lags, corrs)
    except TypeError:
        plt.bar(lags, corrs, width=0.8, align="center")
    plt.axhline(0, linewidth=1)
    plt.title(title)
    plt.xlabel("Lag (months)")
    plt.ylabel("Correlation")
    plt.tight_layout()
    outpath = os.path.join(OUTPUT_DIR_LOCAL, fname)
    plt.savefig(outpath, dpi=600)
    plt.close()

# -------------------- CORE OUTPUTS --------------------
# Tables 01 & 02 are useful even if you only run CORE_FIGS.
totals = df.groupby("sector_group").size().rename("n_postings").reset_index()
min_date, max_date = df[date_col].min(), df[date_col].max()
coverage = pd.DataFrame([{
    "min_date": min_date, "max_date": max_date,
    "n_months": len(m_pivot), "n_quarters": len(q_pivot)
}])
pd.concat([totals, coverage], axis=1).to_csv(os.path.join(OUTPUT_DIR_LOCAL, "Table01_TotalsAndCoverage.csv"), index=False)

diag = q_pivot.apply(basic_stats).T.reset_index().rename(columns={"index":"sector"})
diag.to_csv(os.path.join(OUTPUT_DIR_LOCAL, "Table02_QuarterlyDiagnostics.csv"), index=False)

if DO_CORE_FIGS:
    # Fig01 – Quarterly postings
    save_line_plot(q_pivot[["Construction","Digital","Traditional"]].fillna(0),
                   "Quarterly AI Job Postings by Sector",
                   "Job postings",
                   "Fig01_Quarterly_Postings.png")
    # Fig02 – Growth index (quarterly)
    qi = q_pivot.apply(growth_index)
    save_line_plot(qi[["Construction","Digital","Traditional"]].fillna(0),
                   "Growth Index by Sector (Quarterly, base=first observation)",
                   "Index (Q1=100)",
                   "Fig02_GrowthIndex_Quarterly.png")

# -------------------- STL + Seasonality Summary --------------------
seasonality_rows = []
if DO_STL:
    if HAVE_STATS and len(m_pivot) >= 24:
        for sec in ["Construction","Digital","Traditional"]:
            if sec in m_pivot.columns:
                series = m_pivot[sec].asfreq("MS").fillna(0)
                try:
                    dec = seasonal_decompose(series, model="additive", period=SEAS_PERIODS, extrapolate_trend="freq")
                    S, R = dec.seasonal, dec.resid
                    valid = (~S.isna()) & (~R.isna())
                    var_R = np.var(R[valid]); var_SR = np.var((S[valid] + R[valid]))
                    fs = max(0.0, 1.0 - (var_R/var_SR)) if var_SR>0 else np.nan
                    fig = dec.plot()
                    plt.suptitle(f"Seasonal Decomposition ({sec})", y=1.02)
                    plt.tight_layout()
                    plt.savefig(os.path.join(OUTPUT_DIR_LOCAL, f"Fig0{3 + ['Construction','Digital','Traditional'].index(sec)}_STL_{sec}.png"), dpi=600)
                    plt.close()
                    seasonality_rows.append({"sector": sec, "seasonal_strength": fs})
                except Exception:
                    seasonality_rows.append({"sector": sec, "seasonal_strength": np.nan})
    else:
        tmp = m_pivot.copy()
        tmp["m"] = tmp.index.month
        for sec in ["Construction","Digital","Traditional"]:
            if sec in tmp.columns:
                overall = tmp[sec].mean()
                mm = tmp.groupby("m")[sec].mean()
                ss_between = ((mm - overall)**2).sum()
                ss_total   = ((tmp[sec] - overall)**2).sum()
                fs = (ss_between/ss_total) if ss_total>0 else np.nan
                seasonality_rows.append({"sector": sec, "seasonal_strength_proxy": fs})

# Quarter profile (peak/trough)
q_tmp = q_pivot.copy(); q_tmp["qnum"] = q_tmp.index.quarter
peak_rows = []
for sec in ["Construction","Digital","Traditional"]:
    if sec in q_tmp.columns:
        overall = q_tmp[sec].mean()
        prof = q_tmp.groupby("qnum")[sec].mean() / (overall if overall>0 else 1)
        peak_q   = int(prof.idxmax()) if len(prof)>0 else np.nan
        trough_q = int(prof.idxmin()) if len(prof)>0 else np.nan
        peak_rows.append({"sector": sec, "peak_quarter": peak_q, "trough_quarter": trough_q})

seasonality_df = pd.DataFrame(seasonality_rows) if seasonality_rows else pd.DataFrame(columns=["sector"])
peaks_df       = pd.DataFrame(peak_rows)
tbl3 = pd.merge(seasonality_df, peaks_df, on="sector", how="outer")
tbl3.to_csv(os.path.join(OUTPUT_DIR_LOCAL, "Table03_SeasonalitySummary.csv"), index=False)

# -------------------- Simple CCF (observed) --------------------
if DO_CCF_SIMPLE and "Construction" in m_pivot.columns:
    cc_rows = []
    cons = m_pivot["Construction"].astype(float)
    for other in ["Digital","Traditional"]:
        if other in m_pivot.columns:
            lags, corrs = safe_ccf(cons.values, m_pivot[other].astype(float).values, max_lag=MAX_LAG)
            fname = f"Fig0{6 if other=='Digital' else 7}_CCF_Cons_{other}.png"
            plot_ccf_stem_or_bar(lags, corrs, f"Cross-correlation: Construction vs {other}\n(+lag: {other} leads)", fname)
            idx = int(np.nanargmax(np.abs(corrs)))
            cc_rows.append({
                "pair": f"Construction vs {other}",
                "peak_corr": float(corrs[idx]),
                "lag_at_peak_months": int(lags[idx])
            })
    pd.DataFrame(cc_rows).to_csv(os.path.join(OUTPUT_DIR_LOCAL, "Table04_CrossCorrelation.csv"), index=False)

# -------------------- Ratios (Convergence/Divergence) --------------------
if DO_RATIOS and "Construction" in q_pivot.columns:
    ratio_rows = []
    cons_q = q_pivot["Construction"].replace(0, np.nan)
    for other in ["Digital","Traditional"]:
        if other in q_pivot.columns:
            oth_q = q_pivot[other].replace(0, np.nan)
            ratio = cons_q / oth_q
            x = np.arange(len(ratio)); y = ratio.values
            mask = ~np.isnan(y)
            slope = np.polyfit(x[mask], y[mask], 1)[0] if mask.sum()>=4 else np.nan
            # plot
            plt.figure()
            plt.plot(q_pivot.index, ratio, marker="o")
            plt.title(f"Ratio over time: Construction / {other}")
            plt.xlabel("Quarter"); plt.ylabel("Ratio")
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_DIR_LOCAL, f"Fig0{8 if other=='Digital' else 9}_Ratio_Cons_{other}.png"), dpi=600)
            plt.close()
            ratio_rows.append({
                "ratio": f"Construction/{other}",
                "slope_per_quarter": float(slope) if slope==slope else np.nan,
                "start_ratio": float(pd.Series(ratio).dropna().iloc[0]) if pd.Series(ratio).dropna().size>0 else np.nan,
                "end_ratio": float(pd.Series(ratio).dropna().iloc[-1]) if pd.Series(ratio).dropna().size>0 else np.nan
            })
    pd.DataFrame(ratio_rows).to_csv(os.path.join(OUTPUT_DIR_LOCAL, "Table05_RatioTrends.csv"), index=False)

# -------------------- Rolling ETS (Table06) --------------------
if DO_ETS_ROLLING and HAVE_STATS:
    rows = []
    for sec in ["Construction","Digital","Traditional"]:
        if sec in m_pivot.columns:
            y = m_pivot[sec].astype(float).asfreq("MS").fillna(0)
            if len(y) >= (ROLLING_TEST_MONTHS + 24):
                errs_abs, errs_pct = [], []
                for t in range(ROLLING_TEST_MONTHS, 0, -1):
                    train = y.iloc[: -t]
                    true_next = y.iloc[-t]
                    try:
                        model = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=SEAS_PERIODS)
                        fit = model.fit(optimized=True, use_brute=True)
                        pred = fit.forecast(1).iloc[0]
                        e = abs(pred - true_next)
                        errs_abs.append(e)
                        denom = true_next if true_next != 0 else 1.0
                        errs_pct.append(e / denom)
                    except Exception:
                        pass
                mae  = float(np.mean(errs_abs)) if errs_abs else np.nan
                mape = float(np.mean(errs_pct)) if errs_pct else np.nan
                rows.append({"sector": sec, "rolling_months": ROLLING_TEST_MONTHS, "MAE": mae, "MAPE": mape})
    if rows:
        pd.DataFrame(rows).to_csv(os.path.join(OUTPUT_DIR_LOCAL, "Table06_RollingETS_ForecastAccuracy.csv"), index=False)

# -------------------- Seasonality test (Kruskal–Wallis) --------------------
if DO_KRUSKAL and HAVE_SCIPY:
    tbl7 = []
    qnum = m_pivot.index.to_period("Q").quarter
    for sec in ["Construction","Digital","Traditional"]:
        if sec in m_pivot.columns:
            y = m_pivot[sec].astype(float)
            groups = [y[qnum == q] for q in [1,2,3,4] if (qnum == q).sum() > 0]
            if len(groups) == 4 and all(len(g) > 2 for g in groups):
                H, p = kruskal(*groups)
                n_obs = int(sum(len(g) for g in groups))
                tbl7.append({"sector": sec, "test": "Kruskal-Wallis (quarters)", "H_stat": float(H), "p_value": float(p), "k_groups": 4, "n_obs": n_obs})
    if tbl7:
        pd.DataFrame(tbl7).to_csv(os.path.join(OUTPUT_DIR_LOCAL, "Table07_SeasonalityKruskal.csv"), index=False)

# -------------------- ARIMA vs ETS (figures + Table08) --------------------
def select_arima_order(y, seasonal_periods=12, max_p=1, max_q=1):
    best = (None, None, np.inf, np.inf)
    if not HAVE_STATS: return best
    y = pd.Series(y).astype(float)
    d_candidates = [0, 1]
    D_candidates = [0]   # conservative
    for p in range(0, max_p+1):
        for q in range(0, max_q+1):
            for d in d_candidates:
                for P in [0,1]:
                    for Q in [0,1]:
                        for D in D_candidates:
                            order = (p, d, q)
                            seasonal_order = (P, D, Q, seasonal_periods)
                            try:
                                mod = SARIMAX(y, order=order, seasonal_order=seasonal_order,
                                              enforce_stationarity=False, enforce_invertibility=False)
                                res = mod.fit(disp=False)
                                aic, bic = res.aic, res.bic
                                if aic < best[2]:
                                    best = (order, seasonal_order, aic, bic)
                            except Exception:
                                continue
    return best

def rolling_forecast_arima(y, order, seasonal_order, last_n=8):
    preds, true_vals = [], []
    if not HAVE_STATS or order is None: return np.nan, np.nan, []
    for t in range(last_n, 0, -1):
        train = y.iloc[: -t]
        true_next = y.iloc[-t]
        try:
            mod = SARIMAX(train, order=order, seasonal_order=seasonal_order,
                          enforce_stationarity=False, enforce_invertibility=False)
            res = mod.fit(disp=False)
            pred = res.forecast(1).iloc[0]
            preds.append(pred); true_vals.append(true_next)
        except Exception:
            continue
    if preds:
        errs_abs = [abs(p - v) for p, v in zip(preds, true_vals)]
        errs_pct = [abs(p - v) / (v if v != 0 else 1.0) for p, v in zip(preds, true_vals)]
        return float(np.mean(errs_abs)), float(np.mean(errs_pct)), preds
    else:
        return np.nan, np.nan, []

def rolling_forecast_ets(y, last_n=8):
    preds, true_vals = [], []
    if not HAVE_STATS: return np.nan, np.nan, []
    for t in range(last_n, 0, -1):
        train = y.iloc[: -t]
        true_next = y.iloc[-t]
        try:
            model = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=SEAS_PERIODS)
            fit = model.fit(optimized=True, use_brute=True)
            pred = fit.forecast(1).iloc[0]
            preds.append(pred); true_vals.append(true_next)
        except Exception:
            continue
    if preds:
        errs_abs = [abs(p - v) for p, v in zip(preds, true_vals)]
        errs_pct = [abs(p - v) / (v if v != 0 else 1.0) for p, v in zip(preds, true_vals)]
        return float(np.mean(errs_abs)), float(np.mean(errs_pct)), preds
    else:
        return np.nan, np.nan, []

def plot_forecast_comparison(y, ets_preds, arima_preds, last_n=8, last_n_plot=24, title="", outname=""):
    plt.figure()
    y_tail = y.iloc[-last_n_plot:]
    plt.plot(y_tail.index, y_tail.values, label="Actual")
    if ets_preds and len(ets_preds) == last_n:
        ets_idx = y.index[-last_n:]
        plt.plot(ets_idx, ets_preds, marker="o", linestyle="--", label="ETS (1-step)")
    if arima_preds and len(arima_preds) == last_n:
        arima_idx = y.index[-last_n:]
        plt.plot(arima_idx, arima_preds, marker="s", linestyle="--", label="ARIMA (1-step)")
    plt.title(title)
    plt.xlabel("Time"); plt.ylabel("Job postings")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR_LOCAL, outname), dpi=600)
    plt.close()

if DO_ARIMA_VS_ETS and HAVE_STATS:
    tbl8 = []
    for sec, figname in [("Construction", "Fig10_Forecasts_Construction.png"),
                         ("Digital", "Fig11_Forecasts_Digital.png"),
                         ("Traditional", "Fig12_Forecasts_Traditional.png")]:
        if sec in m_pivot.columns:
            y = m_pivot[sec].astype(float).asfreq("MS").fillna(0)
            if len(y) >= (ROLLING_TEST_MONTHS + 24):
                train0 = y.iloc[:-ROLLING_TEST_MONTHS]
                order, seas_order, aic, bic = select_arima_order(train0, seasonal_periods=SEAS_PERIODS,
                                                                 max_p=ARIMA_MAX_P, max_q=ARIMA_MAX_Q)
                mae_ets, mape_ets, ets_preds = rolling_forecast_ets(y, last_n=ROLLING_TEST_MONTHS)
                mae_ar,  mape_ar,  ar_preds  = rolling_forecast_arima(y, order, seas_order, last_n=ROLLING_TEST_MONTHS)
                tbl8.append({"sector": sec, "model": "ETS",   "MAE": mae_ets, "MAPE": mape_ets, "AIC": np.nan, "BIC": np.nan})
                tbl8.append({"sector": sec, "model": "ARIMA", "MAE": mae_ar,  "MAPE": mape_ar,  "AIC": aic,    "BIC": bic})
                plot_forecast_comparison(y, ets_preds, ar_preds, last_n=ROLLING_TEST_MONTHS,
                                         last_n_plot=24,
                                         title=f"{sec}: Rolling One-step Forecasts (ETS vs ARIMA)",
                                         outname=figname)
    if tbl8:
        pd.DataFrame(tbl8).to_csv(os.path.join(OUTPUT_DIR_LOCAL, "Table08_ForecastModelComparison.csv"), index=False)

# -------------------- Bootstrap CCF with CI --------------------
def circular_block_bootstrap_indices(n, block_len, rng):
    idx = []
    while len(idx) < n:
        start = rng.integers(0, n)
        block = [(start + k) % n for k in range(block_len)]
        idx.extend(block)
    return np.array(idx[:n])

def bootstrap_ccf_ci(x, y, max_lag=6, n_boot=1500, block_len=6, random_state=123):
    rng = np.random.default_rng(random_state)
    x = pd.Series(x).astype(float).reset_index(drop=True)
    y = pd.Series(y).astype(float).reset_index(drop=True)
    lags, obs = safe_ccf(x.values, y.values, max_lag)
    boot_mat = np.empty((n_boot, len(lags)))
    for b in range(n_boot):
        idx = circular_block_bootstrap_indices(len(x), block_len, rng)
        xb, yb = x.iloc[idx].reset_index(drop=True), y.iloc[idx].reset_index(drop=True)
        _, cb = safe_ccf(xb.values, yb.values, max_lag)
        boot_mat[b, :] = cb
    lower = np.nanpercentile(boot_mat, 2.5, axis=0)
    upper = np.nanpercentile(boot_mat, 97.5, axis=0)
    return lags, obs, lower, upper

def plot_ccf_with_ci(lags, obs, lo, hi, title, outname):
    plt.figure()
    plt.plot(lags, obs, marker="o", linestyle="-", label="Observed CCF")
    plt.fill_between(lags, lo, hi, alpha=0.2, label="Bootstrap 95% CI")
    plt.axhline(0, linewidth=1)
    plt.title(title)
    plt.xlabel("Lag (months)")
    plt.ylabel("Correlation")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR_LOCAL, outname), dpi=600)
    plt.close()

if DO_BOOTSTRAP_CCF and "Construction" in m_pivot.columns:
    tbl9 = []
    cons = m_pivot["Construction"].astype(float).values
    for other, tag in [("Digital","Digital"), ("Traditional","Trad")]:
        if other in m_pivot.columns:
            oth = m_pivot[other].astype(float).values
            lags, obs, lo, hi = bootstrap_ccf_ci(cons, oth, max_lag=MAX_LAG, n_boot=N_BOOT, block_len=BLOCK_LEN)
            outname = f"Fig1{3 if tag=='Digital' else 4}_CCF_Bootstrap_Cons_{tag}.png"
            plot_ccf_with_ci(lags, obs, lo, hi, f"Bootstrap CCF: Construction vs {other}", outname)
            idx = int(np.nanargmax(np.abs(obs)))
            tbl9.append({
                "pair": f"Construction vs {other}",
                "peak_corr": float(obs[idx]),
                "lag_at_peak_months": int(lags[idx]),
                "ci_lower_at_peak": float(lo[idx]),
                "ci_upper_at_peak": float(hi[idx]),
                "ci_includes_zero": bool(lo[idx] <= 0.0 <= hi[idx])
            })
            # Per-lag CI bands
            pd.DataFrame({"lag": lags, "obs_corr": obs, "ci_lower": lo, "ci_upper": hi}) \
              .to_csv(os.path.join(OUTPUT_DIR_LOCAL, f"Table09_BootstrapCCF_CIBands_{tag}.csv"), index=False)
    if tbl9:
        pd.DataFrame(tbl9).to_csv(os.path.join(OUTPUT_DIR_LOCAL, "Table09_BootstrapCCF_Summary.csv"), index=False)

# -------------------- COPY EVERYTHING TO DRIVE ONCE --------------------
os.makedirs(OUTPUT_DIR_DRIVE, exist_ok=True)
for name in os.listdir(OUTPUT_DIR_LOCAL):
    src = os.path.join(OUTPUT_DIR_LOCAL, name)
    dst = os.path.join(OUTPUT_DIR_DRIVE, name)
    try:
        shutil.copy2(src, dst)
    except Exception as e:
        print("Copy failed for", name, ":", e)

print("\nAll outputs copied to:", OUTPUT_DIR_DRIVE)
print("Figures expected:",
      "Fig01..Fig14; Tables 01..09 (+ per-lag CI files).")


In [ ]:
# =========================
# Revision Helper v2 (auto-detect monthly data; no heavy rerun)
# - Finds monthly series in memory or CSVs; last resort: quick rebuild from SOURCE_XLSX if defined
# - Computes: STL seasonal-strength + Month-of-Year KW with MBB p-values
# - Forecasts: stable ETS-only (MAE, MASE, sMAPE); ARIMA excluded
# - Saves CSVs; prints paste-ready snippets
# =========================
import os, glob, re, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")

# ---- Optional: point to a ready monthly CSV if detection fails ----
OPTIONAL_MONTHLY_CSV_PATH = globals().get("OPTIONAL_MONTHLY_CSV_PATH", None)

# ---- Notebook-level paths/params (use existing if present) ----
OUTPUT_DIR_LOCAL = globals().get("OUTPUT_DIR_LOCAL", "YOUR_PATH")
OUTPUT_DIR_DRIVE = globals().get("OUTPUT_DIR_DRIVE", None)
ROLLING_TEST_MONTHS = int(globals().get("ROLLING_TEST_MONTHS", 8))
BLOCK_LEN = int(globals().get("BLOCK_LEN", 6))
SEAS_PERIODS = int(globals().get("SEAS_PERIODS", 12))
os.makedirs(OUTPUT_DIR_LOCAL, exist_ok=True)

# ---- Imports for methods ----
try:
    from statsmodels.tsa.seasonal import STL
    HAVE_STL = True
except Exception:
    HAVE_STL = False
from scipy.stats import kruskal

# ---------- Data detection helpers ----------
def _looks_wide(df):
    cols = set([c.lower() for c in df.columns])
    want = {"construction","digital","traditional"}
    return len(want & cols) >= 2  # at least two expected sectors

def _to_monthly_index(idx):
    try:
        di = pd.to_datetime(idx)
        return di.to_period("M").to_timestamp()
    except Exception:
        return None

def _standardise_tidy(df):
    cols = {c.lower(): c for c in df.columns}
    date = next((cols[k] for k in ["date","month","dt","time","period"] if k in cols), None)
    sector = next((cols[k] for k in ["sector","industry","group","category"] if k in cols), None)
    posts = next((cols[k] for k in ["postings","count","n","value","jobs","job_postings"] if k in cols), None)
    if not (date and sector and posts): return None
    out = df[[date,sector,posts]].rename(columns={date:"date", sector:"sector", posts:"postings"}).copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.to_period("M").dt.to_timestamp()
    out["postings"] = pd.to_numeric(out["postings"], errors="coerce")
    out = out.dropna(subset=["date","postings"])
    piv = out.pivot_table(index="date", columns="sector", values="postings", aggfunc="sum").sort_index()
    return piv

def _from_globals():
    # Preferred: m_pivot already built in the notebook
    cand = globals().get("m_pivot", None)
    if isinstance(cand, pd.DataFrame):
        idx = _to_monthly_index(cand.index)
        if idx is not None:
            df = cand.copy()
            df.index = idx
            return df.sort_index()
    # Other likely names
    for name, obj in list(globals().items()):
        if isinstance(obj, pd.DataFrame) and len(obj) >= 24:
            df = obj.copy()
            # wide?
            if _looks_wide(df):
                idx = _to_monthly_index(df.index)
                if idx is not None:
                    df.index = idx
                    return df.sort_index()
            # tidy?
            piv = _standardise_tidy(df)
            if isinstance(piv, pd.DataFrame) and len(piv)>=24:
                return piv
    return None

def _from_csv(path):
    try:
        if path.lower().endswith((".parquet",".pq")):
            df = pd.read_parquet(path)
        else:
            df = pd.read_csv(path)
    except Exception:
        return None
    # wide?
    if _looks_wide(df):
        if "date" in [c.lower() for c in df.columns]:
            dcol = [c for c in df.columns if c.lower()=="date"][0]
            df[dcol] = pd.to_datetime(df[dcol], errors="coerce").dt.to_period("M").dt.to_timestamp()
            df = df.set_index(dcol)
        if df.index.dtype != "datetime64[ns]":
            idx = _to_monthly_index(df.index)
            if idx is not None:
                df.index = idx
        return df.sort_index()
    # tidy?
    piv = _standardise_tidy(df)
    return piv

def _search_files(base):
    if not base or not os.path.exists(base): return None
    pats = ["*m_pivot*.csv","*monthly*pivot*.csv","*monthly*sector*.csv","*postings*monthly*.csv","*month*counts*.csv"]
    for pat in pats:
        for fp in glob.glob(os.path.join(base, pat)):
            piv = _from_csv(fp)
            if isinstance(piv, pd.DataFrame) and len(piv)>=24:
                print(f"[Found] monthly data at {fp}")
                return piv
    return None

def _quick_from_xlsx():
    src = globals().get("SOURCE_XLSX", None)
    if not src or not os.path.exists(src): return None
    try:
        x = pd.ExcelFile(src)
        sheet = next((s for s in x.sheet_names if s.lower() in {"data","all","merged","postings"}), x.sheet_names[0])
        df = pd.read_excel(src, sheet_name=sheet)
    except Exception:
        return None
    # Try to build a small monthly pivot (lightweight)
    date_col = next((c for c in df.columns if "date" in c.lower()), None)
    if date_col is None:
        for c in df.columns:
            try:
                pd.to_datetime(df[c]); date_col = c; break
            except: pass
    if date_col is None: return None
    # sector grouping: try an existing sector column first; else derive from ANZSIC-like numeric
    sec_col = next((c for c in df.columns if any(k in c.lower() for k in ["sector","industry","group"])), None)
    if sec_col is None:
        # derive from any numeric-like ANZSIC column
        num_col = next((c for c in df.columns if re.search(r"anz|code|industry", c.lower())), None)
        if num_col is None: return None
        def _to_num(v):
            try: return int(re.findall(r"\d+", str(v))[0])
            except: return np.nan
        code = df[num_col].apply(_to_num)
        def _grp(code):
            if pd.isna(code): return "Unknown"
            code = int(code)
            if 3000 <= code <= 3999: return "Construction"
            if (5800 <= code <= 6399) or (6900 <= code <= 7299): return "Digital"
            return "Traditional"
        df["__sector__"] = code.apply(_grp)
        sec_col = "__sector__"
    df["__date__"] = pd.to_datetime(df[date_col], errors="coerce").dt.to_period("M").dt.to_timestamp()
    g = df.groupby(["__date__", sec_col]).size().rename("postings").reset_index()
    piv = g.pivot(index="__date__", columns=sec_col, values="postings").sort_index()
    return piv

# ---------- Load monthly pivot ----------
monthly = _from_globals()
if monthly is None and OPTIONAL_MONTHLY_CSV_PATH and os.path.exists(OPTIONAL_MONTHLY_CSV_PATH):
    monthly = _from_csv(OPTIONAL_MONTHLY_CSV_PATH)
if monthly is None:
    monthly = _search_files(OUTPUT_DIR_LOCAL)
if monthly is None and OUTPUT_DIR_DRIVE:
    monthly = _search_files(OUTPUT_DIR_DRIVE)
if monthly is None:
    monthly = _quick_from_xlsx()

if monthly is None:
    raise RuntimeError(
        "Monthly data not found. Options:\n"
        "  1) Run earlier cells that create m_pivot, OR\n"
        "  2) Set OPTIONAL_MONTHLY_CSV_PATH to a monthly CSV (tidy or wide), OR\n"
        "  3) Ensure SOURCE_XLSX is defined and accessible."
    )

# Keep only expected sectors present
SECTORS = [s for s in ["Construction","Digital","Traditional"] if s in monthly.columns]
if not SECTORS:
    # If custom labels exist (e.g., 'Const'), allow a soft map
    soft_map = {}
    for c in monthly.columns:
        cl = c.lower()
        if 'construct' in cl: soft_map[c] = 'Construction'
        elif 'digit' in cl or 'ict' in cl: soft_map[c] = 'Digital'
        elif 'tradit' in cl or 'other' in cl or 'rest' in cl: soft_map[c] = 'Traditional'
    if soft_map:
        monthly = monthly.rename(columns=soft_map)
        SECTORS = [s for s in ["Construction","Digital","Traditional"] if s in monthly.columns]
if not SECTORS:
    raise RuntimeError("Could not map sector columns. Rename columns to include Construction/Digital/Traditional (or set OPTIONAL_MONTHLY_CSV_PATH).")

monthly = monthly.asfreq("MS").sort_index()
print(f"[OK] Monthly data loaded. Rows={len(monthly)}, Columns={list(monthly.columns)}")

# ---------- Seasonality: STL strength + MoY KW with MBB ----------
def seasonal_strength_STL(y, period=12):
    if not HAVE_STL:
        raise RuntimeError("statsmodels STL not available in this runtime.")
    res = STL(y, period=period, robust=True).fit()
    S, R = res.seasonal, res.resid
    vR = np.var(R.dropna(), ddof=1); vSR = np.var((S+R).dropna(), ddof=1)
    fs = max(0.0, 1 - (vR / vSR)) if vSR>0 else np.nan
    return fs, res

def _mbb_indices(n, block_len, rng):
    starts = rng.integers(0, n, size=int(np.ceil(n/block_len)))
    idx = np.concatenate([np.arange(s, s+block_len) for s in starts])[:n] % n
    return idx

def kw_mbb_pval(detrended, block_len=6, B=1500, seed=7):
    rng = np.random.default_rng(seed)
    x = detrended.values
    mo = detrended.index.month.values
    uniq = np.unique(mo)
    groups = [x[mo==m] for m in uniq]
    H_obs, _ = kruskal(*groups)
    n = len(x)
    H_boot = np.empty(B)
    for b in range(B):
        idx = _mbb_indices(n, block_len, rng)
        mo_bs = mo[idx]
        groups_bs = [x[mo_bs==m] for m in uniq]
        H_boot[b], _ = kruskal(*groups_bs)
    p_val = (np.sum(H_boot >= H_obs) + 1) / (B + 1)
    return float(H_obs), float(p_val)

season_rows = []
stl_res = {}
for sec in SECTORS:
    y = monthly[sec].astype(float).interpolate(limit_direction="both")
    fs, res = seasonal_strength_STL(y, period=SEAS_PERIODS)
    stl_res[sec] = res
    detr = (res.observed - res.trend).dropna()
    H, p = kw_mbb_pval(detr, block_len=BLOCK_LEN, B=1500, seed=7)
    season_rows.append({"sector": sec, "n_months": int(len(y)), "seasonal_strength": round(fs,3), "KW_H_mbb": round(H,2), "KW_p_mbb": p})

seasonality_tbl = pd.DataFrame(season_rows).sort_values("sector")
season_path = os.path.join(OUTPUT_DIR_LOCAL, "Table07b_Seasonality_MonthOfYear.csv")
seasonality_tbl.to_csv(season_path, index=False)

# ---------- Forecasts: stable ETS-only (rolling one-step) ----------
from statsmodels.tsa.holtwinters import ExponentialSmoothing
def sMAPE(y, yhat, eps=1e-9):
    y = np.asarray(y); yhat = np.asarray(yhat)
    return 100 * np.mean(np.abs(y - yhat) / ((np.abs(y) + np.abs(yhat) + eps)/2))
def MASE(y, yhat, seasonality=12):
    y = np.asarray(y); yhat = np.asarray(yhat)
    if len(y) <= seasonality: return np.nan
    denom = np.mean(np.abs(y[seasonality:] - y[:-seasonality]))
    if denom == 0: return np.nan
    return np.mean(np.abs(y - yhat)) / denom

fmetrics = []
for sec in SECTORS:
    y = monthly[sec].astype(float).fillna(0.0)
    if len(y) < (ROLLING_TEST_MONTHS + 24):
        continue
    preds, actuals = [], []
    ok_points = 0
    for t in range(ROLLING_TEST_MONTHS, 0, -1):
        train = y.iloc[: -t]
        true_next = y.iloc[-t]
        try:
            mod = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=SEAS_PERIODS)
            fit = mod.fit(optimized=True)
            pred = float(fit.forecast(1).iloc[0])
            preds.append(pred); actuals.append(float(true_next))
            ok_points += 1
        except Exception:
            pass
    if ok_points:
        mae   = float(np.mean(np.abs(np.array(actuals) - np.array(preds))))
        mase  = float(MASE(np.array(actuals), np.array(preds), seasonality=SEAS_PERIODS))
        smape = float(sMAPE(np.array(actuals), np.array(preds)))
        fmetrics.append({"sector": sec, "model": "ETS (stable-only)", "roll_steps": ok_points,
                         "MAE": mae, "MASE": mase, "sMAPE": smape})

forecast_tbl = pd.DataFrame(fmetrics).sort_values("sector")
forecast_path = os.path.join(OUTPUT_DIR_LOCAL, "Table08b_ForecastMetrics_Stable.csv")
forecast_tbl.to_csv(forecast_path, index=False)

# ---------- Optional mirror to Drive ----------
if OUTPUT_DIR_DRIVE:
    try:
        os.makedirs(OUTPUT_DIR_DRIVE, exist_ok=True)
        import shutil
        for p in [season_path, forecast_path]:
            if os.path.exists(p):
                shutil.copy2(p, os.path.join(OUTPUT_DIR_DRIVE, os.path.basename(p)))
    except Exception as e:
        print("Drive mirror skipped:", e)

# ---------- Paste-ready snippets ----------
def _fmtp(p):
    if pd.isna(p): return "n/a"
    return f"{p:.3f}" if p >= 0.001 else "<0.001"

methods_txt = (
    f"We adopt additive STL (period={SEAS_PERIODS}) on monthly series to report seasonal-strength (variance share) "
    f"and evaluate month-of-year differences on STL trend-removed values using a moving-block bootstrap Kruskal–Wallis "
    f"test (block={BLOCK_LEN} months; 1,500 replications). Forecast accuracy is reported for stable ETS one-step models; "
    f"ARIMA is excluded to avoid instability."
)
season_lines = ["Seasonality (STL strength; MoY KW MBB p-values):"]
for _, r in seasonality_tbl.iterrows():
    season_lines.append(f" - {r['sector']}: strength={r['seasonal_strength']:.3f}; MoY p={_fmtp(r['KW_p_mbb'])}")
timing_txt = ("Cross-sector timing: bootstrap CCF bands in our analysis include zero at candidate lags; "
              "we do not detect statistically reliable timing relationships.")
forecast_txt = ""
if not forecast_tbl.empty:
    rows = [f" - {r['sector']}: MAE={r['MAE']:.2f}, MASE={r['MASE']:.2f}, sMAPE={r['sMAPE']:.1f}%"
            for _, r in forecast_tbl.iterrows()]
    forecast_txt = "Forecasting (one-step, stable ETS only):\n" + "\n".join(rows)

print("\n========== COPY INTO MANUSCRIPT ==========")
print("\nMethods addendum:\n" + methods_txt)
print("\nSeasonality summary (main text numbers):\n" + "\n".join(season_lines))
print("\nTiming conclusion (discussion):\n" + timing_txt)
if forecast_txt:
    print("\nForecasting paragraph:\n" + forecast_txt)
print("==========================================\n")
print(f"[Saved] {season_path}")
print(f"[Saved] {forecast_path}")


In [ ]:
# --- MASE fix: compute seasonal-naïve denominator from full monthly series and update Table08b ---
import os, numpy as np, pandas as pd

OUTPUT_DIR_LOCAL = globals().get("OUTPUT_DIR_LOCAL", "YOUR_PATH")
SEAS_PERIODS = int(globals().get("SEAS_PERIODS", 12))

# Load monthly pivot (already in notebook) and forecast table from disk
if 'monthly' in globals():
    _monthly = monthly.copy()
elif 'm_pivot' in globals() and isinstance(m_pivot, pd.DataFrame):
    _monthly = m_pivot.copy()
else:
    raise RuntimeError("Monthly data not found (monthly/m_pivot). Run the previous helper first.")

fc_path = os.path.join(OUTPUT_DIR_LOCAL, "Table08b_ForecastMetrics_Stable.csv")
fc = pd.read_csv(fc_path)

# Build seasonal-naïve denominators per sector from the FULL series
def seasonal_naive_denom(y, s=12):
    y = np.asarray(y, dtype=float)
    if len(y) <= s: return np.nan
    return np.mean(np.abs(y[s:] - y[:-s]))

denoms = {}
for col in _monthly.columns:
    y = _monthly[col].astype(float).values
    denoms[col] = seasonal_naive_denom(y, s=SEAS_PERIODS)

# Update MASE = MAE / denom (sector-wise)
fc['MASE'] = fc.apply(lambda r: (r['MAE'] / denoms.get(r['sector'], np.nan)) if denoms.get(r['sector'], np.nan) not in (0, np.nan) else np.nan, axis=1)

# Save back
fc.to_csv(fc_path, index=False)
print("Updated Table08b with MASE using full-series seasonal-naïve denominator:")
print(fc.to_string(index=False))


# Step 3.4 & 4.6: Bootstrap Cross-Correlation on STL Remainders (B = 1,500)

Cross-correlations computed on STL remainders with circular moving-block bootstrap (B = 1,500, block = 6, seed = 123), matching manuscript Section 3.4, Section 4.6, Figures 14–15, Table 09, and Supplementary Table S4.

In [ ]:
# ============================================================
# Bootstrap Cross-Correlation on STL Remainders (B = 1,500)
# Matches manuscript Section 3.4 / 4.6 & Figures 14–15 (Table 09 / Table S4)
# ============================================================

import os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL

# Use notebook parameters / relative paths (no hard-coded machine paths)
DATA = globals().get("SOURCE_XLSX", "data/all_industries_merged.xlsx")
OUT  = globals().get("OUTPUT_DIR_LOCAL", "outputs")
os.makedirs(OUT, exist_ok=True)

B_BOOT  = 1500
BLOCK   = 6
MAX_LAG = 6
SEED    = 123

# Check if monthly pivot already exists in memory; if not, load from DATA
if "m_pivot" in globals() and isinstance(globals()["m_pivot"], pd.DataFrame):
    m_counts = globals()["m_pivot"].copy()
    print("Using existing m_pivot from notebook memory.")
else:
    print(f"Loading data from: {DATA}")
    df = pd.read_excel(DATA)
    
    def _sector(code):
        try: c = int(code)
        except: return None
        if 3000 <= c <= 3999: return "Construction"
        if (5800 <= c <= 6399) or (6900 <= c <= 7299): return "Digital"
        return "Traditional"

    date_col = [c for c in df.columns if 'date' in c.lower() or 'posting' in c.lower()][0]
    anzsic_col = [c for c in df.columns if 'anzsic' in c.lower() or 'industry' in c.lower()][0]

    df['Sector'] = df[anzsic_col].apply(_sector)
    df = df[df['Sector'].notna()].copy()
    df['Date'] = pd.to_datetime(df[date_col])
    df['Month'] = df['Date'].dt.to_period('M').dt.to_timestamp()

    m_counts = df.groupby(['Month', 'Sector']).size().unstack(fill_value=0).sort_index()

# ------------------------------------------------------------------
# STL remainders
# ------------------------------------------------------------------
def stl_remainder(s, period=12, robust=True):
    res = STL(s.astype(float), period=period, robust=robust).fit()
    return pd.Series(res.resid, index=s.index)

remainders = pd.DataFrame({
    sec: stl_remainder(m_counts[sec], period=12, robust=True)
    for sec in ['Construction', 'Digital', 'Traditional'] if sec in m_counts.columns
})
print(f"STL remainders computed. Shape: {remainders.shape}")

# ------------------------------------------------------------------
# Sample CCF
# ------------------------------------------------------------------
def ccf(x, y, max_lag=MAX_LAG):
    """Returns dict lag -> Pearson r for corr(x_t, y_{t+L})."""
    n = len(x)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x = (x - x.mean())
    y = (y - y.mean())
    sx = np.sqrt((x ** 2).sum())
    sy = np.sqrt((y ** 2).sum())
    out = {}
    for L in range(-max_lag, max_lag + 1):
        if L >= 0:
            num = np.dot(x[:n - L], y[L:n])
        else:
            num = np.dot(x[-L:n], y[:n + L])
        out[L] = float(num / (sx * sy))
    return out

# ------------------------------------------------------------------
# Moving-block bootstrap on remainders
# ------------------------------------------------------------------
def bootstrap_ccf_ci(x, y, B=B_BOOT, block=BLOCK, max_lag=MAX_LAG, seed=SEED):
    """Joint block bootstrap on (x, y) pairs; computes CCF on each replicate."""
    n = len(x)
    rng = np.random.default_rng(seed)
    obs = ccf(x, y, max_lag=max_lag)
    pairs = np.column_stack([np.asarray(x), np.asarray(y)])
    boot = {L: np.empty(B) for L in obs}
    for b in range(B):
        L_arr = len(pairs)
        n_blocks = int(np.ceil(n / block))
        starts = rng.integers(0, L_arr, size=n_blocks)
        idx = np.concatenate([(np.arange(block) + s) % L_arr for s in starts])[:n]
        boot_pairs = pairs[idx]
        bx, by = boot_pairs[:, 0], boot_pairs[:, 1]
        c = ccf(bx, by, max_lag=max_lag)
        for L in c:
            boot[L][b] = c[L]
    rows = []
    for L in sorted(obs):
        lo, hi = np.percentile(boot[L], [2.5, 97.5])
        rows.append({
            'lag': L,
            'obs_r': obs[L],
            'ci_low_95': lo,
            'ci_high_95': hi,
            'significant_95': not (lo <= 0 <= hi),
        })
    return pd.DataFrame(rows)

print(f"\nRunning bootstrap CCF (B={B_BOOT}, block={BLOCK}, seed={SEED}) on STL remainders…")
cons = remainders['Construction'].dropna().values
dig  = remainders['Digital'].dropna().values
trad = remainders['Traditional'].dropna().values
n = min(len(cons), len(dig), len(trad))
cons, dig, trad = cons[:n], dig[:n], trad[:n]

ccf_dig  = bootstrap_ccf_ci(cons, dig,  B=B_BOOT)
ccf_trad = bootstrap_ccf_ci(cons, trad, B=B_BOOT)

print("\nConstruction–Digital (STL remainders, B=1500):")
print(ccf_dig.round(4).to_string(index=False))
print("\nConstruction–Traditional (STL remainders, B=1500):")
print(ccf_trad.round(4).to_string(index=False))

# Save Table09 CSVs
ccf_dig.to_csv(os.path.join(OUT, 'Table09_BootstrapCCF_CIBands_Digital.csv'), index=False)
ccf_trad.to_csv(os.path.join(OUT, 'Table09_BootstrapCCF_CIBands_Trad.csv'), index=False)

# Summary
def sig_lags(df):
    s = df[df['significant_95']]
    return "; ".join(f"lag {int(r.lag):+d}: r={r.obs_r:.3f} [{r.ci_low_95:.3f},{r.ci_high_95:.3f}]"
                     for _, r in s.iterrows()) or "none"

summary = pd.DataFrame([
    {'pair': 'Construction–Digital',     'method': 'STL remainders', 'B': B_BOOT, 'block': BLOCK,
     'series_length': n, 'significant_lags_95CI': sig_lags(ccf_dig)},
    {'pair': 'Construction–Traditional', 'method': 'STL remainders', 'B': B_BOOT, 'block': BLOCK,
     'series_length': n, 'significant_lags_95CI': sig_lags(ccf_trad)},
])
summary.to_csv(os.path.join(OUT, 'Table09_BootstrapCCF_Summary.csv'), index=False)
print(f"\nWrote: Table09_BootstrapCCF_CIBands_Digital.csv")
print(f"Wrote: Table09_BootstrapCCF_CIBands_Trad.csv")
print(f"Wrote: Table09_BootstrapCCF_Summary.csv")

# ------------------------------------------------------------------
# Figures 14 & 15 (Figures 13 & 14 in raw script, mapped to manuscript)
# ------------------------------------------------------------------
def ccf_plot(ccf_df, title, out_path):
    fig, ax = plt.subplots(figsize=(9, 5))
    lags = ccf_df['lag'].values
    r    = ccf_df['obs_r'].values
    lo   = ccf_df['ci_low_95'].values
    hi   = ccf_df['ci_high_95'].values
    ax.bar(lags, r, color=['#c0392b' if s else '#7f8c8d' for s in ccf_df['significant_95']],
           alpha=0.85, width=0.6)
    ax.errorbar(lags, r, yerr=[r-lo, hi-r], fmt='none', ecolor='black', capsize=4, lw=1)
    ax.axhline(0, color='k', lw=0.6)
    ax.set_xlabel('Lag (months)')
    ax.set_ylabel('Pearson r')
    ax.set_title(title)
    ax.set_xticks(lags)
    ax.grid(True, alpha=0.3, axis='y')
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    print(f"Wrote: {out_path}")

ccf_plot(ccf_dig,
         f"Bootstrap CCF (Construction vs Digital, STL remainders, B={B_BOOT})\nRed bars = 95% CI excludes 0",
         os.path.join(OUT, 'Fig14_CCF_Bootstrap_Cons_Digital.png'))
ccf_plot(ccf_trad,
         f"Bootstrap CCF (Construction vs Traditional, STL remainders, B={B_BOOT})\nRed bars = 95% CI excludes 0",
         os.path.join(OUT, 'Fig15_CCF_Bootstrap_Cons_Trad.png'))

print("CCF on STL remainders complete.")